#Initial work

In [ ]:
!pip install prince
!pip install kmodes
!pip install ydata-profiling
!pip install numpy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mahdimashayekhi/social-media-vs-productivity")

print("Path to dataset files:", path)

In [ ]:
df_original = pd.read_csv(os.path.join(path, os.listdir(path)[0]))

In [ ]:
display(df_original.head())

In [ ]:
df_original.info()

In [ ]:
print(f"Number of NaNs in 'actual_productivity_score' before dropping: {df_original['actual_productivity_score'].isnull().sum()}")

df_original.dropna(subset=['actual_productivity_score'], inplace=True)

df_original.info()

In [ ]:
df_original.loc[df_original["weekly_offline_hours"] == 0, "weekly_offline_hours"] = np.nan


In [ ]:
display(df_original.isnull().sum()[df_original.isnull().sum() > 0])

In [ ]:
missing_data = df_original.isnull().sum()
missing_data = missing_data[missing_data > 0]

missing_df = pd.DataFrame({
    'Feature': missing_data.index,
    'Missing Count': missing_data.values,
    'Missing Percentage': (missing_data.values / len(df_original)) * 100
})

missing_df = missing_df.sort_values(by='Missing Count', ascending=False).reset_index(drop=True)
display(missing_df)

In [ ]:
display(df_original.describe())

##ydata-profiling report

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df_original, title="Pandas Profiling Report", explorative = True)
profile.to_file("report.html")

##train set evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_original.drop(columns=["actual_productivity_score", "perceived_productivity_score"])
y = df_original["actual_productivity_score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
nans_df = pd.DataFrame({
    'NaN Count': X_train.isnull().sum(),
    'NaN Percentage': (X_train.isnull().sum() / len(X_train)) * 100
})

display(nans_df[nans_df['NaN Count'] > 0].sort_values(by='NaN Count', ascending=False))

#Imputing

##Median

In [ ]:
# Make copies so original data is preserved
X_train_median = X_train.copy()
X_test_median = X_test.copy()

In [ ]:
num_cols_with_nan = [
    'weekly_offline_hours',
    'daily_social_media_time',
    'job_satisfaction_score',
    'sleep_hours',
    'screen_time_before_sleep',
    'stress_level'
]

# Optional: print missing percentages
print(X_train_median[num_cols_with_nan].isna().mean() * 100)

In [ ]:
for col in num_cols_with_nan:
    median_value = X_train_median[col].median()
    X_train_median[col] = X_train_median[col].fillna(median_value)
    X_test_median[col] = X_test_median[col].fillna(median_value)  # use train median for test

In [ ]:
print(X_train_median[num_cols_with_nan].isna().sum())
print(X_test_median[num_cols_with_nan].isna().sum())

In [ ]:
import math

num_cols_with_nan = [
    'weekly_offline_hours',
    'daily_social_media_time',
    'job_satisfaction_score',
    'sleep_hours',
    'screen_time_before_sleep',
    'stress_level'
]

n_features = len(num_cols_with_nan)
n_comparisons_per_row = 2 # Number of feature comparisons per row
n_cols_per_comparison = 2 # Before and Median Imputation

n_rows = math.ceil(n_features / n_comparisons_per_row)
n_cols = n_comparisons_per_row * n_cols_per_comparison # 2 comparisons * 2 plots each = 4 columns

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes = axes.flatten() # Flatten axes array for easier indexing

for i, col in enumerate(num_cols_with_nan):
    # Calculate the correct subplot index for the current feature
    row_idx = i // n_comparisons_per_row
    col_start_idx = (i % n_comparisons_per_row) * n_cols_per_comparison

    # Before imputation
    ax_before = axes[row_idx * n_cols + col_start_idx]
    sns.histplot(X_train[col], bins=30, kde=True, color='skyblue', ax=ax_before)
    ax_before.set_title(f'{col} - Before Imputation')
    ax_before.set_ylabel('Frequency')

    # Median imputation
    ax_after = axes[row_idx * n_cols + col_start_idx + 1]
    sns.histplot(X_train_median[col], bins=30, kde=True, color='lightgreen', ax=ax_after)
    ax_after.set_title(f'{col} - Median Imputation')
    ax_after.set_ylabel('Frequency')

# Hide any unused subplots if n_features is odd
for j in range(i * n_cols_per_comparison + n_cols_per_comparison, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

##KNN

In [ ]:
# Copies for KNN imputation
X_train_knn = X_train.copy()
X_test_knn = X_test.copy()

In [ ]:
# -------------------------------
# 1️⃣ Identify columns by type
# -------------------------------
numeric_cols = X_train_knn.select_dtypes(include=['int64', 'float64']).columns.tolist()
bool_cols = X_train_knn.select_dtypes(include=['bool']).columns.tolist()
cat_cols = X_train_knn.select_dtypes(include=['object']).columns.tolist()

# -------------------------------
# 2️⃣ Convert bools to int
# -------------------------------
X_train_knn[bool_cols] = X_train_knn[bool_cols].astype(int)
X_test_knn[bool_cols] = X_test_knn[bool_cols].astype(int)

# -------------------------------
# 3️⃣ One-Hot Encode categorical columns
# -------------------------------
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit on train, transform both train & test
X_train_ohe = pd.DataFrame(
    ohe.fit_transform(X_train_knn[cat_cols]),
    columns=ohe.get_feature_names_out(cat_cols),
    index=X_train_knn.index
)

X_test_ohe = pd.DataFrame(
    ohe.transform(X_test_knn[cat_cols]),
    columns=ohe.get_feature_names_out(cat_cols),
    index=X_test_knn.index
)

# -------------------------------
# 4️⃣ Combine all features for KNN
# -------------------------------
# Numeric + bools + OHE categorical columns
X_train_knn_full = pd.concat([X_train_knn[numeric_cols + bool_cols], X_train_ohe], axis=1)
X_test_knn_full = pd.concat([X_test_knn[numeric_cols + bool_cols], X_test_ohe], axis=1)

# -------------------------------
# 5️⃣ Scale numeric + bool columns
# -------------------------------
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_train_knn_full[numeric_cols + bool_cols] = scaler.fit_transform(X_train_knn_full[numeric_cols + bool_cols])
X_test_knn_full[numeric_cols + bool_cols] = scaler.transform(X_test_knn_full[numeric_cols + bool_cols])

# -------------------------------
# 6️⃣ Ready for KNN imputation
# -------------------------------
all_features_for_knn = X_train_knn_full.columns.tolist()

In [ ]:
num_cols_with_nan = [
    'weekly_offline_hours',
    'daily_social_media_time',
    'job_satisfaction_score',
    'sleep_hours',
    'screen_time_before_sleep',
    'stress_level'
]

In [ ]:
from sklearn.impute import KNNImputer

# Initialize KNN imputer
knn_imputer = KNNImputer(
    n_neighbors=5,      # number of neighbors
    weights='uniform',  # simple average
    metric='nan_euclidean'  # handles missing values in neighbors
)

# Fit on train and transform both train & test
X_train_knn_full[num_cols_with_nan] = knn_imputer.fit_transform(X_train_knn_full[num_cols_with_nan])
X_test_knn_full[num_cols_with_nan] = knn_imputer.transform(X_test_knn_full[num_cols_with_nan])

In [ ]:
print(X_train_knn_full[num_cols_with_nan].isna().sum())
print(X_test_knn_full[num_cols_with_nan].isna().sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

num_cols_with_nan = [
    'weekly_offline_hours',
    'daily_social_media_time',
    'job_satisfaction_score',
    'sleep_hours',
    'screen_time_before_sleep',
    'stress_level'
]

In [ ]:
n_features = len(num_cols_with_nan)
n_cols_per_feature = 2 # Before and After KNN Imputation
fig, axes = plt.subplots(n_features, n_cols_per_feature, figsize=(15, 4 * n_features))

for i, col in enumerate(num_cols_with_nan):
    # After KNN imputation (second column of subplots) - Plot first to get its y-limits
    ax_after = axes[i, 1]
    sns.histplot(X_train_knn_full[col], bins=30, kde=True, color='salmon', ax=ax_after)
    ax_after.set_title(f'{col} - After KNN Imputation')
    ax_after.set_ylabel('Frequency')

    # Capture y-axis limits from the 'After' plot
    y_min, y_max = ax_after.get_ylim()

    # Before imputation (first column of subplots)
    ax_before = axes[i, 0]
    sns.histplot(X_train_knn[col], bins=30, kde=True, color='skyblue', ax=ax_before)
    ax_before.set_title(f'{col} - Before Imputation')
    ax_before.set_ylabel('Frequency')

    # Set the same y-axis limits for both plots
    ax_before.set_ylim(y_min, y_max)
    ax_after.set_ylim(y_min, y_max)

plt.tight_layout()
plt.show()

##MICE

In [ ]:
X_train_mice = X_train.copy()
X_test_mice = X_test.copy()

In [ ]:
numeric_cols = X_train_mice.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

bool_cols = X_train_mice.select_dtypes(
    include=['bool']
).columns.tolist()

cat_cols = X_train_mice.select_dtypes(
    include=['object']
).columns.tolist()

In [ ]:
X_train_mice[bool_cols] = X_train_mice[bool_cols].astype(int)
X_test_mice[bool_cols] = X_test_mice[bool_cols].astype(int)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

ohe = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

X_train_ohe = pd.DataFrame(
    ohe.fit_transform(X_train_mice[cat_cols]),
    columns=ohe.get_feature_names_out(cat_cols),
    index=X_train_mice.index
)

X_test_ohe = pd.DataFrame(
    ohe.transform(X_test_mice[cat_cols]),
    columns=ohe.get_feature_names_out(cat_cols),
    index=X_test_mice.index
)

In [ ]:
X_train_mice_full = pd.concat(
    [X_train_mice[numeric_cols + bool_cols], X_train_ohe],
    axis=1
)

X_test_mice_full = pd.concat(
    [X_test_mice[numeric_cols + bool_cols], X_test_ohe],
    axis=1
)

In [ ]:
from sklearn.preprocessing import RobustScaler

scale_cols = numeric_cols + bool_cols

scaler = RobustScaler()

X_train_mice_full[scale_cols] = scaler.fit_transform(
    X_train_mice_full[scale_cols]
)

X_test_mice_full[scale_cols] = scaler.transform(
    X_test_mice_full[scale_cols]
)

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

mice_imputer = IterativeImputer(
    max_iter=5,
    initial_strategy="median",
    random_state=42
)

X_train_mice_imputed = mice_imputer.fit_transform(X_train_mice_full)
X_test_mice_imputed = mice_imputer.transform(X_test_mice_full)

In [ ]:
X_train_mice_imputed = pd.DataFrame(
    X_train_mice_imputed,
    columns=X_train_mice_full.columns,
    index=X_train.index
)

X_test_mice_imputed = pd.DataFrame(
    X_test_mice_imputed,
    columns=X_test_mice_full.columns,
    index=X_test.index
)

In [ ]:
for col in num_cols_with_nan:

    plt.figure(figsize=(12,5))

    # BEFORE MICE
    plt.subplot(1,2,1)
    sns.histplot(
        X_train[col],
        bins=30,
        kde=True
    )
    plt.title(f"{col} - Before MICE Imputation")

    # AFTER MICE
    plt.subplot(1,2,2)
    sns.histplot(
        X_train_mice_imputed[col],
        bins=30,
        kde=True
    )
    plt.title(f"{col} - After MICE Imputation")

    plt.tight_layout()
    plt.show()

In [ ]:
# Undo scaling
X_train_mice_unscaled = X_train_mice_imputed.copy()

X_train_mice_unscaled[scale_cols] = scaler.inverse_transform(
    X_train_mice_imputed[scale_cols]
)

In [ ]:
for col in num_cols_with_nan:

    plt.figure(figsize=(8,5))

    sns.kdeplot(X_train[col], label="Before", fill=True)
    sns.kdeplot(X_train_mice_unscaled[col], label="After MICE", fill=True)

    plt.title(f"{col} Distribution Comparison")
    plt.legend()
    plt.show()

In [ ]:
print(X_train[col].var())
print(X_train_mice_unscaled[col].var())

In [ ]:
# Convert MICE output to DataFrame
X_train_mice_imputed = pd.DataFrame(
    X_train_mice_imputed,
    columns=X_train_mice_full.columns,
    index=X_train_mice_full.index
)

In [ ]:
import matplotlib.pyplot as plt

nan_numeric_cols = [
    col for col in X_train_mice_full.columns
    if X_train_mice_full[col].isna().sum() > 0
]

print("Features with missing values:")
print(nan_numeric_cols)


n_features_to_plot = len(nan_numeric_cols)
n_cols = 3
n_rows = (n_features_to_plot + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(nan_numeric_cols):

    ax = axes[i]

    X_train_mice_full[col].dropna().hist(
        bins=40,
        alpha=0.5,
        label="Original",
        ax=ax
    )

    X_train_mice_imputed[col].hist(
        bins=40,
        alpha=0.5,
        label="MICE Imputed",
        ax=ax
    )

    ax.set_title(col)
    ax.legend()

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

##MICE - Bayesian

In [ ]:
# =============================
# Bayesian MICE (IterativeImputer + BayesianRidge)
# with:
# - bool -> int
# - OneHot for object cats
# - RobustScaler on numeric+bool only
# - Impute in scaled space
# - Inverse-scale back
# - FIX discrete/ordinal stress_level by round+clip+astype(int)
# =============================

import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

# -----------------------------
# Inputs assumed to exist:
# X_train, X_test (pandas DataFrames)
# -----------------------------

# -----------------------------
# 1) Copy
# -----------------------------
X_train_bmice = X_train.copy()
X_test_bmice = X_test.copy()

# -----------------------------
# 2) Detect column types
# -----------------------------
numeric_cols = X_train_bmice.select_dtypes(include=["int64", "float64"]).columns.tolist()
bool_cols = X_train_bmice.select_dtypes(include=["bool"]).columns.tolist()
cat_cols = X_train_bmice.select_dtypes(include=["object"]).columns.tolist()

# -----------------------------
# 3) Bool -> int (so scaler & imputer can handle)
# -----------------------------
if len(bool_cols) > 0:
    X_train_bmice[bool_cols] = X_train_bmice[bool_cols].astype(int)
    X_test_bmice[bool_cols] = X_test_bmice[bool_cols].astype(int)

# -----------------------------
# 4) One-hot encode categorical (object) columns
# -----------------------------
if len(cat_cols) > 0:
    ohe = OneHotEncoder(
        sparse_output=False,
        handle_unknown="ignore"
    )

    X_train_ohe = pd.DataFrame(
        ohe.fit_transform(X_train_bmice[cat_cols]),
        columns=ohe.get_feature_names_out(cat_cols),
        index=X_train_bmice.index
    )

    X_test_ohe = pd.DataFrame(
        ohe.transform(X_test_bmice[cat_cols]),
        columns=ohe.get_feature_names_out(cat_cols),
        index=X_test_bmice.index
    )
else:
    # No categorical columns
    ohe = None
    X_train_ohe = pd.DataFrame(index=X_train_bmice.index)
    X_test_ohe = pd.DataFrame(index=X_test_bmice.index)

# -----------------------------
# 5) Combine numeric/bool + OHE
# -----------------------------
base_cols = numeric_cols + bool_cols
X_train_bmice_full = pd.concat([X_train_bmice[base_cols], X_train_ohe], axis=1)
X_test_bmice_full = pd.concat([X_test_bmice[base_cols], X_test_ohe], axis=1)

# -----------------------------
# 6) Scale numeric+bool only (NOT OHE)
# -----------------------------
scale_cols = base_cols  # numeric + bool

scaler_bmice = RobustScaler()
if len(scale_cols) > 0:
    X_train_bmice_full.loc[:, scale_cols] = scaler_bmice.fit_transform(X_train_bmice_full[scale_cols])
    X_test_bmice_full.loc[:, scale_cols] = scaler_bmice.transform(X_test_bmice_full[scale_cols])

# -----------------------------
# 7) Bayesian MICE Imputation
# -----------------------------
bayesian_mice = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=10,
    initial_strategy="median",
    sample_posterior=True,
    random_state=42
)

X_train_bmice_imputed_arr = bayesian_mice.fit_transform(X_train_bmice_full)
X_test_bmice_imputed_arr = bayesian_mice.transform(X_test_bmice_full)

X_train_bmice_imputed = pd.DataFrame(
    X_train_bmice_imputed_arr,
    columns=X_train_bmice_full.columns,
    index=X_train_bmice_full.index
)

X_test_bmice_imputed = pd.DataFrame(
    X_test_bmice_imputed_arr,
    columns=X_test_bmice_full.columns,
    index=X_test_bmice_full.index
)

# -----------------------------
# 8) Inverse-scale numeric+bool back to original scale
# -----------------------------
X_train_bmice_unscaled = X_train_bmice_imputed.copy()
X_test_bmice_unscaled = X_test_bmice_imputed.copy()

if len(scale_cols) > 0:
    X_train_bmice_unscaled.loc[:, scale_cols] = scaler_bmice.inverse_transform(X_train_bmice_imputed[scale_cols])
    X_test_bmice_unscaled.loc[:, scale_cols] = scaler_bmice.inverse_transform(X_test_bmice_imputed[scale_cols])

# -----------------------------
# 9) FIX: stress_level is discrete/ordinal but got continuous values
#    -> round + clip to allowed integer range
#    EDIT THESE to match your dataset.
# -----------------------------
STRESS_COL = "stress_level"
STRESS_MIN, STRESS_MAX = 1, 10   # <-- change to (0,10) etc. to match your scale

if STRESS_COL in X_train_bmice_unscaled.columns:
    for df in [X_train_bmice_unscaled, X_test_bmice_unscaled]:
        df[STRESS_COL] = (
            df[STRESS_COL]
            .round()
            .clip(STRESS_MIN, STRESS_MAX)
            .astype(int)
        )

# -----------------------------
# 10) Optional: return bool cols back to bool (if you want)
# -----------------------------
# If you want bools as True/False again:
# for df in [X_train_bmice_unscaled, X_test_bmice_unscaled]:
#     for c in bool_cols:
#         if c in df.columns:
#             df[c] = df[c].round().clip(0, 1).astype(int).astype(bool)

# -----------------------------
# OUTPUTS:
# - X_train_bmice_unscaled
# - X_test_bmice_unscaled
# - ohe (encoder), scaler_bmice, bayesian_mice
# -----------------------------

print("Done.")
print("Shapes:")
print("X_train_bmice_unscaled:", X_train_bmice_unscaled.shape)
print("X_test_bmice_unscaled:", X_test_bmice_unscaled.shape)
if STRESS_COL in X_train_bmice_unscaled.columns:
    print("stress_level unique (train, first 20):", np.sort(X_train_bmice_unscaled[STRESS_COL].unique())[:20])

###diagnotic

In [ ]:
missing_masks = {
    col: X_train[col].isna()
    for col in num_cols_with_nan
}

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(nan_numeric_cols):

    ax = axes[i]

    X_train_bmice_full[col].dropna().hist(
        bins=40,
        alpha=0.5,
        label="Original",
        ax=ax
    )

    X_train_bmice_imputed[col].hist(
        bins=40,
        alpha=0.5,
        label="Bayesian MICE Imputed",
        ax=ax
    )

    ax.set_title(col)
    ax.legend()

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# If X_train_bmice_imputed is currently a numpy array:
X_train_bmice_imputed_df = pd.DataFrame(
    X_train_bmice_imputed,
    columns=X_train_bmice_full.columns,
    index=X_train_bmice_full.index
)

In [ ]:
X_train_bmice_imputed_df = pd.DataFrame(
    X_train_bmice_imputed,
    columns=X_train_bmice_full.columns,
    index=X_train_bmice_full.index
)

In [ ]:
features_imputed = [
    col for col in X_train_bmice_full.columns
    if X_train_bmice_full[col].isna().sum() > 0
]

features_imputed

In [ ]:
from scipy.stats import ks_2samp

ks_rows = []

for col in features_imputed:

    before = X_train_bmice_full[col].dropna()
    after = X_train_bmice_imputed_df[col]

    stat, p = ks_2samp(before, after)

    interpretation = (
        "Distribution changed"
        if p < 0.05 else
        "No significant distribution change"
    )

    ks_rows.append({
        "Feature": col,
        "KS Statistic": round(stat,4),
        "p-value": round(p,4),
        "Interpretation": interpretation
    })

ks_table = pd.DataFrame(ks_rows)

ks_table

In [ ]:
from scipy.stats import anderson_ksamp

ad_rows = []

for col in features_imputed:

    before = X_train_bmice_full[col].dropna()
    after = X_train_bmice_imputed_df[col]

    result = anderson_ksamp([before, after])

    stat = result.statistic
    p = result.significance_level / 100

    interpretation = (
        "Distribution changed"
        if p < 0.05 else
        "No significant distribution change"
    )

    ad_rows.append({
        "Feature": col,
        "AD Statistic": round(stat,4),
        "p-value": round(p,4),
        "Interpretation": interpretation
    })

ad_table = pd.DataFrame(ad_rows)

ad_table

In [ ]:
from scipy.stats import mannwhitneyu

mw_rows = []

for col in features_imputed:

    before = X_train_bmice_full[col].dropna()
    after = X_train_bmice_imputed_df[col]

    stat, p = mannwhitneyu(before, after)

    interpretation = (
        "Median changed"
        if p < 0.05 else
        "Median preserved"
    )

    mw_rows.append({
        "Feature": col,
        "MWU Statistic": round(stat,4),
        "p-value": round(p,4),
        "Interpretation": interpretation
    })

mw_table = pd.DataFrame(mw_rows)

mw_table

In [ ]:
from scipy.stats import levene

bf_rows = []

for col in features_imputed:

    before = X_train_bmice_full[col].dropna()
    after = X_train_bmice_imputed_df[col]

    stat, p = levene(before, after, center="median")

    interpretation = (
        "Variance changed"
        if p < 0.05 else
        "Variance preserved"
    )

    bf_rows.append({
        "Feature": col,
        "BF Statistic": round(stat,4),
        "p-value": round(p,4),
        "Interpretation": interpretation
    })

bf_table = pd.DataFrame(bf_rows)

bf_table

In [ ]:
import numpy as np

def cliffs_delta(x, y):

    x = np.array(x)
    y = np.array(y)

    gt = (x[:,None] > y).mean()
    lt = (x[:,None] < y).mean()

    return gt - lt


cd_rows = []

for col in features_imputed:

    before = X_train_bmice_full[col].dropna()
    after = X_train_bmice_imputed_df[col]

    delta = cliffs_delta(before.values, after.values)

    abs_d = abs(delta)

    if abs_d < 0.147:
        interpretation = "Negligible difference"
    elif abs_d < 0.33:
        interpretation = "Small difference"
    elif abs_d < 0.474:
        interpretation = "Medium difference"
    else:
        interpretation = "Large difference"

    cd_rows.append({
        "Feature": col,
        "Cliff's Delta": round(delta,4),
        "Interpretation": interpretation
    })

cd_table = pd.DataFrame(cd_rows)

cd_table

In [ ]:
pd.set_option("display.max_colwidth", None)

summary_rows = []

total_features = len(ks_table)


# Kolmogorov–Smirnov
ks_sig = (ks_table["p-value"] < 0.05).sum()

summary_rows.append({
    "Test": "Kolmogorov–Smirnov",
    "Features tested": total_features,
    "Significant differences": ks_sig,
    "Interpretation":
        "Distribution preserved"
        if ks_sig == 0 else
        "Some distribution differences"
})


# Anderson–Darling
ad_sig = (ad_table["p-value"] < 0.05).sum()

summary_rows.append({
    "Test": "Anderson–Darling",
    "Features tested": total_features,
    "Significant differences": ad_sig,
    "Interpretation":
        "Highly sensitive; effect size should be considered"
})


# Mann–Whitney U
mw_sig = (mw_table["p-value"] < 0.05).sum()

summary_rows.append({
    "Test": "Mann–Whitney U",
    "Features tested": total_features,
    "Significant differences": mw_sig,
    "Interpretation":
        "Median preserved"
        if mw_sig == 0 else
        "Median changed in some features"
})


# Brown–Forsythe
bf_sig = (bf_table["p-value"] < 0.05).sum()

summary_rows.append({
    "Test": "Brown–Forsythe",
    "Features tested": total_features,
    "Significant differences": bf_sig,
    "Interpretation":
        "Variance preserved"
        if bf_sig == 0 else
        "Variance changed in some features"
})


# Cliff's Delta
cd_sig = (cd_table["Interpretation"] != "Negligible difference").sum()

summary_rows.append({
    "Test": "Cliff's Delta",
    "Features tested": total_features,
    "Significant differences": cd_sig,
    "Interpretation":
        "No meaningful practical difference"
        if cd_sig == 0 else
        "Practical difference exists"
})


summary_table = pd.DataFrame(summary_rows)

summary_table

##MissForest

In [ ]:
import pandas as pd
import numpy as np
import gc

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# =====================================================
# 1) COPY DATA
# =====================================================
X_train_mf = X_train.copy()
X_test_mf = X_test.copy()

# =====================================================
# 2) SEPARATE ORDINAL TARGET FOR CLASSIFICATION
# =====================================================
ordinal_col = "stress_level"
stress_train = X_train_mf[ordinal_col].copy()
stress_test = X_test_mf[ordinal_col].copy()

X_train_mf.drop(columns=[ordinal_col], inplace=True)
X_test_mf.drop(columns=[ordinal_col], inplace=True)

# =====================================================
# 3) IDENTIFY COLUMN TYPES (BEFORE ANY CAST)
# =====================================================
numeric_cols = X_train_mf.select_dtypes(include=["int64", "float64"]).columns.tolist()
bool_cols = X_train_mf.select_dtypes(include=["bool"]).columns.tolist()
cat_cols = X_train_mf.select_dtypes(include=["object"]).columns.tolist()

# =====================================================
# 4) LABEL ENCODE CATEGORICAL VARIABLES (keep numeric for models)
# =====================================================
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()

    X_train_mf[col] = X_train_mf[col].astype(str)
    X_test_mf[col] = X_test_mf[col].astype(str)

    X_train_mf[col] = le.fit_transform(X_train_mf[col])

    unknown_val = len(le.classes_)
    X_test_mf[col] = X_test_mf[col].map(
        lambda x: le.transform([x])[0] if x in le.classes_ else unknown_val
    )

    label_encoders[col] = le

# =====================================================
# 5) BOOLEAN -> SMALL INTEGER (RAM saver)
# =====================================================
X_train_mf[bool_cols] = X_train_mf[bool_cols].astype(np.int8)
X_test_mf[bool_cols] = X_test_mf[bool_cols].astype(np.int8)

# =====================================================
# 6) CAST WHOLE MATRIX TO float32 (after encoding)
# =====================================================
X_train_mf = X_train_mf.astype(np.float32)
X_test_mf  = X_test_mf.astype(np.float32)

gc.collect()

# =====================================================
# 7) MISSFOREST-LIKE IMPUTATION (REGRESSION PART) - RAM OPTIMIZED
# =====================================================
imputer = IterativeImputer(
    estimator=RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        max_samples=0.7,
        max_features="sqrt",
        n_jobs=1,              # IMPORTANT for Colab RAM
        random_state=42
    ),
    max_iter=3,               # IMPORTANT for Colab RAM
    random_state=42,
    initial_strategy="most_frequent",
    skip_complete=True
)

X_train_mf = pd.DataFrame(
    imputer.fit_transform(X_train_mf),
    columns=X_train_mf.columns,
    index=X_train.index
)
X_test_mf = pd.DataFrame(
    imputer.transform(X_test_mf),
    columns=X_test_mf.columns,
    index=X_test.index
)

print("✅ After RF regression-imputation - missing train:", X_train_mf.isna().sum().sum())
print("✅ After RF regression-imputation - missing test :", X_test_mf.isna().sum().sum())

gc.collect()

# =====================================================
# 8) TRUE MISSFOREST STEP: IMPUTE stress_level VIA CLASSIFICATION
# =====================================================
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    max_samples=0.8,
    max_features="sqrt",
    n_jobs=1,              # IMPORTANT for Colab RAM (avoid -1)
    random_state=42
)

observed_mask = stress_train.notna()
rf_clf.fit(X_train_mf.loc[observed_mask], stress_train.loc[observed_mask])

# Train: fill missing + keep observed
missing_mask = stress_train.isna()
X_train_mf.loc[missing_mask, ordinal_col] = rf_clf.predict(X_train_mf.loc[missing_mask])
X_train_mf.loc[~missing_mask, ordinal_col] = stress_train.loc[~missing_mask]

# Test: fill missing + keep observed
missing_mask_test = stress_test.isna()
X_test_mf.loc[missing_mask_test, ordinal_col] = rf_clf.predict(X_test_mf.loc[missing_mask_test])
X_test_mf.loc[~missing_mask_test, ordinal_col] = stress_test.loc[~missing_mask_test]

# Make sure stress_level is integer (discrete)
X_train_mf[ordinal_col] = X_train_mf[ordinal_col].astype(int)
X_test_mf[ordinal_col] = X_test_mf[ordinal_col].astype(int)

# =====================================================
# 9) DECODE CATEGORICAL VARIABLES (NOW SAFE)
# =====================================================
for col in cat_cols:
    le = label_encoders[col]

    X_train_mf[col] = (
        X_train_mf[col]
        .round()
        .clip(0, len(le.classes_) - 1)
        .astype(int)
    )
    X_test_mf[col] = (
        X_test_mf[col]
        .round()
        .clip(0, len(le.classes_) - 1)
        .astype(int)
    )

    X_train_mf[col] = le.inverse_transform(X_train_mf[col])
    X_test_mf[col]  = le.inverse_transform(X_test_mf[col])

# =====================================================
# 10) RESTORE BOOLEAN TYPE
# =====================================================
X_train_mf[bool_cols] = X_train_mf[bool_cols].astype(bool)
X_test_mf[bool_cols]  = X_test_mf[bool_cols].astype(bool)

# =====================================================
# ✅ FINAL CHECKS
# =====================================================
print("✅ Final missing values (Train):", X_train_mf.isna().sum().sum())
print("✅ Final missing values (Test):", X_test_mf.isna().sum().sum())
print("✅ Stress level unique values (Train):", sorted(pd.unique(X_train_mf[ordinal_col])))

###diagnostic

In [ ]:
print("Train missing values:")
print(X_train_mf.isna().sum().sort_values(ascending=False).head())

print("\nTest missing values:")
print(X_test_mf.isna().sum().sort_values(ascending=False).head())

In [ ]:
import matplotlib.pyplot as plt

nan_numeric_cols = [
    col for col in numeric_cols
    if X_train[col].isna().sum() > 0
]

# add stress_level manually
if "stress_level" in X_train.columns:
    nan_numeric_cols.append("stress_level")

print("Features with missing values:")
print(nan_numeric_cols)


n_features_to_plot = len(nan_numeric_cols)
n_cols = 3
n_rows = (n_features_to_plot + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(nan_numeric_cols):

    ax = axes[i]

    X_train[col].dropna().hist(
        bins=40,
        alpha=0.5,
        label="Original",
        ax=ax
    )

    X_train_mf[col].hist(
        bins=40,
        alpha=0.5,
        label="Imputed",
        ax=ax
    )

    ax.set_title(col)
    ax.legend()


for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import ks_2samp

results = []

for col in numeric_cols:
    stat, p = ks_2samp(
        X_train[col].dropna(),
        X_train_mf[col]
    )
    results.append([col, p])

ks_results = pd.DataFrame(results, columns=["Feature","p_value"])
display(ks_results.sort_values("p_value"))

In [ ]:
diag_stats = pd.DataFrame({
    "orig_mean": X_train[numeric_cols].mean(),
    "imp_mean": X_train_mf[numeric_cols].mean(),
    "orig_std": X_train[numeric_cols].std(),
    "imp_std": X_train_mf[numeric_cols].std()
})

diag_stats["mean_diff_%"] = (
    abs(diag_stats.orig_mean - diag_stats.imp_mean)
    / abs(diag_stats.orig_mean)
) * 100

diag_stats.sort_values("mean_diff_%", ascending=False)

In [ ]:
nan_cols = [col for col in X_train.columns if X_train[col].isna().sum() > 0]

results = []

for col in nan_cols:

    missing_pct = X_train[col].isna().mean()

    median_val = X_train[col].median()

    missing_mask = X_train[col].isna()

    imputed_equal_median = (
        (X_train_mf.loc[missing_mask, col] == median_val)
        .sum()
    )

    total_imputed = missing_mask.sum()

    pct_equal_median = (
        imputed_equal_median / total_imputed
        if total_imputed > 0 else 0
    )

    results.append({
        "Column": col,
        "Missing_%": round(missing_pct*100, 2),
        "Median": median_val,
        "Imputed_Count": total_imputed,
        "Equal_to_Median_Count": imputed_equal_median,
        "Equal_to_Median_%": round(pct_equal_median*100, 2)
    })

    mean_val = X_train[col].mean()

    imputed_equal_mean = (
        (X_train_mf.loc[missing_mask, col].round(3) == round(mean_val,3))
    ).sum()


results_df = pd.DataFrame(results)

results_df.sort_values(
    "Equal_to_Median_%",
    ascending=False,
    inplace=True
)

results_df.reset_index(drop=True, inplace=True)

results_df

In [ ]:
import numpy as np

# columns that had NaNs originally
nan_cols = [col for col in X_train.columns if X_train[col].isna().sum() > 0]

# remove stress_level
nan_cols = [col for col in nan_cols if col != "stress_level"]

# apply only to continuous numeric columns
continuous_nan_cols = [
    col for col in nan_cols
    if np.issubdtype(X_train[col].dtype, np.number)
]

# add noise
for col in continuous_nan_cols:

    missing_mask = X_train[col].isna()

    if missing_mask.sum() == 0:
        continue

    std = X_train[col].std()

    noise = np.random.normal(
        loc=0,
        scale=std * 0.1,   # 10% noise level
        size=missing_mask.sum()
    )

    X_train_mf.loc[missing_mask, col] += noise


print("Noise added to columns:")
print(continuous_nan_cols)

In [ ]:
import matplotlib.pyplot as plt

nan_numeric_cols = [
    col for col in numeric_cols
    if X_train[col].isna().sum() > 0
]

# add stress_level manually
if "stress_level" in X_train.columns:
    nan_numeric_cols.append("stress_level")

print("Features with missing values:")
print(nan_numeric_cols)


n_features_to_plot = len(nan_numeric_cols)
n_cols = 3
n_rows = (n_features_to_plot + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(nan_numeric_cols):

    ax = axes[i]

    X_train[col].dropna().hist(
        bins=40,
        alpha=0.5,
        label="Original",
        ax=ax
    )

    X_train_mf[col].hist(
        bins=40,
        alpha=0.5,
        label="Imputed",
        ax=ax
    )

    ax.set_title(col)
    ax.legend()


for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

##PMM

In [ ]:
# =========================
# 0) Install + imports
# =========================
!pip -q install miceforest

In [ ]:
import numpy as np
import pandas as pd
import miceforest as mf

# 1) Safe copies
X_train_pmm = X_train.copy(deep=True)
X_test_pmm  = X_test.copy(deep=True)

# Save original indices
train_index = X_train_pmm.index
test_index  = X_test_pmm.index

# miceforest needs RangeIndex
X_train_pmm = X_train_pmm.reset_index(drop=True)
X_test_pmm  = X_test_pmm.reset_index(drop=True)

# 2) Dtypes
obj_cols = X_train_pmm.select_dtypes(include="object").columns
for col in obj_cols:
    X_train_pmm[col] = X_train_pmm[col].astype("category")
    X_test_pmm[col]  = X_test_pmm[col].astype("category")

bool_cols = X_train_pmm.select_dtypes(include="bool").columns
for col in bool_cols:
    X_train_pmm[col] = X_train_pmm[col].astype("int8")
    X_test_pmm[col]  = X_test_pmm[col].astype("int8")

if "stress_level" in X_train_pmm.columns:
    X_train_pmm["stress_level"] = X_train_pmm["stress_level"].astype("category")
    X_test_pmm["stress_level"]  = X_test_pmm["stress_level"].astype("category")

# 3) Kernel (PMM is controlled by mean_match_candidates/strategy)
kernel = mf.ImputationKernel(
    data=X_train_pmm,
    num_datasets=1,
    mean_match_candidates=5,
    mean_match_strategy="normal",
    save_all_iterations_data=True,
    random_state=42
)

# 4) Run MICE/PMM on TRAIN
kernel.mice(iterations=5)
X_train_pmm_imp = kernel.complete_data(dataset=0)

# 5) Impute TEST using the trained models (NO .mice() here)
imputed_test = kernel.impute_new_data(
    new_data=X_test_pmm,
    iterations=5,
    save_all_iterations_data=True,
    random_state=42
)
X_test_pmm_imp = imputed_test.complete_data(dataset=0)

# 6) Restore original indices
X_train_pmm_imp.index = train_index
X_test_pmm_imp.index  = test_index

X_train_pmm = X_train_pmm_imp
X_test_pmm  = X_test_pmm_imp

# 7) Final checks
print("Missing Train:", X_train_pmm.isna().sum().sum())
print("Missing Test :", X_test_pmm.isna().sum().sum())

###diagnostics

In [ ]:
print("Train missing:", X_train_pmm.isna().sum().sum())
print("Test missing:", X_test_pmm.isna().sum().sum())

In [ ]:
import matplotlib.pyplot as plt

nan_cols = [col for col in X_train.columns if X_train[col].isna().sum() > 0]

n_cols = 3
n_rows = (len(nan_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5*n_rows))
axes = axes.flatten()

for i, col in enumerate(nan_cols):

    ax = axes[i]

    X_train[col].dropna().hist(
        bins=40,
        alpha=0.5,
        label="Original",
        ax=ax
    )

    X_train_pmm[col].hist(
        bins=40,
        alpha=0.5,
        label="PMM",
        ax=ax
    )

    ax.set_title(col)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

# Redefine nan_cols and nan_numeric_cols to ensure they are available
nan_cols = [col for col in X_train.columns if X_train[col].isna().sum() > 0]
nan_numeric_cols = [col for col in nan_cols if pd.api.types.is_numeric_dtype(X_train[col])]

rows = []

for col in nan_numeric_cols:
    mask = X_train[col].isna()
    if mask.sum() == 0:
        continue

    # original std (numeric)
    orig_std = pd.to_numeric(X_train[col], errors="coerce").std()

    # PMM std on the originally-missing rows (force numeric for computation)
    pmm_vals = pd.to_numeric(X_train_pmm.loc[mask, col], errors="coerce")
    pmm_std = pmm_vals.std()

    rows.append({
        "Column": col,
        "Imputed_Count": int(mask.sum()),
        "Orig_STD": float(orig_std) if pd.notna(orig_std) else np.nan,
        "PMM_Imputed_STD": float(pmm_std) if pd.notna(pmm_std) else np.nan,
        "Imputed_STD_Ratio": float(pmm_std / orig_std) if (pd.notna(pmm_std) and pd.notna(orig_std) and orig_std != 0) else np.nan
    })

diag_df = pd.DataFrame(rows).sort_values("Imputed_STD_Ratio")
diag_df

In [ ]:
col = "stress_level"

freq_compare = pd.DataFrame({
    "Original_%": X_train[col].value_counts(normalize=True, dropna=True),
    "PMM_%": X_train_pmm[col].value_counts(normalize=True, dropna=True)
}).fillna(0)

freq_compare["Abs_Diff"] = (freq_compare["Original_%"] - freq_compare["PMM_%"]).abs()
freq_compare.sort_index()

In [ ]:
numeric_cols = X_train.select_dtypes(include=np.number).columns

corr_before = X_train[numeric_cols].corr()
corr_after = X_train_pmm[numeric_cols].corr()

corr_diff = (corr_before - corr_after).abs()

print("Mean correlation change:",
      corr_diff.mean().mean())

In [ ]:
results = []

for col in nan_cols:

    median = X_train[col].median()

    mask = X_train[col].isna()

    pct = (
        (X_train_pmm.loc[mask, col] == median)
        .mean() * 100
    )

    results.append([col, pct])

pd.DataFrame(results,
             columns=["Column","% equal median"])

In [ ]:
import numpy as np
import pandas as pd

nan_cols = [c for c in X_train.columns if X_train[c].isna().sum() > 0]
nan_numeric_cols = [c for c in nan_cols if np.issubdtype(X_train[c].dtype, np.number)]

# coerce PMM numeric columns to numeric for computation only
X_train_pmm_num = X_train_pmm[nan_numeric_cols].apply(pd.to_numeric, errors="coerce")

compare = pd.DataFrame({
    "Original": X_train[nan_numeric_cols].var(),
    "MissForest": X_train_mf[nan_numeric_cols].var(),
    "PMM": X_train_pmm_num.var()
})

compare["MF/Orig"] = compare["MissForest"] / compare["Original"]
compare["PMM/Orig"] = compare["PMM"] / compare["Original"]

compare.sort_values("PMM/Orig")

In [ ]:
import numpy as np
import pandas as pd

from scipy.stats import ks_2samp, mannwhitneyu, levene
from scipy.stats import anderson_ksamp

def iqr(x):
    return np.nanpercentile(x, 75) - np.nanpercentile(x, 25)

def cliffs_delta(x, y):
    # Nonparametric effect size; approximate via ranks to avoid O(n^2)
    # delta = P(X>Y) - P(X<Y)
    x = np.asarray(x)
    y = np.asarray(y)
    x = x[~np.isnan(x)]
    y = y[~np.isnan(y)]
    if len(x)==0 or len(y)==0:
        return np.nan
    # rank-based approximation using broadcasting is too expensive; sample if huge
    max_n = 4000
    rng = np.random.RandomState(42)
    if len(x) > max_n:
        x = rng.choice(x, size=max_n, replace=False)
    if len(y) > max_n:
        y = rng.choice(y, size=max_n, replace=False)
    gt = (x[:, None] > y[None, :]).mean()
    lt = (x[:, None] < y[None, :]).mean()
    return gt - lt

def pmm_diagnostic_tests(X_before, X_after, numeric_cols, min_imputed=30):
    rows = []
    for col in numeric_cols:
        miss = X_before[col].isna()
        n_imp = int(miss.sum())
        n_obs = int((~miss).sum())
        if n_imp < min_imputed:
            continue

        obs = X_before.loc[~miss, col].astype(float).to_numpy()
        imp = X_after.loc[miss, col].astype(float).to_numpy()

        # Basic ratios
        var_ratio = np.nanvar(imp, ddof=1) / np.nanvar(obs, ddof=1) if np.nanvar(obs, ddof=1) > 0 else np.nan
        iqr_ratio = iqr(imp) / iqr(obs) if iqr(obs) > 0 else np.nan

        # KS (distribution)
        ks_p = ks_2samp(obs, imp, alternative="two-sided", mode="auto").pvalue

        # Anderson-Darling k-sample (distribution, tail sensitive)
        # returns significance_level in %
        try:
            ad_res = anderson_ksamp([obs, imp])
            ad_p = ad_res.significance_level / 100.0
        except Exception:
            ad_p = np.nan

        # Mann-Whitney (location/ranks)
        mw_p = mannwhitneyu(obs, imp, alternative="two-sided").pvalue

        # Brown–Forsythe variance test (Levene centered at median)
        bf_p = levene(obs, imp, center="median").pvalue

        # Effect size
        cd = cliffs_delta(obs, imp)

        rows.append({
            "feature": col,
            "n_obs": n_obs,
            "n_imputed": n_imp,
            "KS_p": ks_p,
            "AD_p": ad_p,
            "MWU_p": mw_p,
            "BrownForsythe_p": bf_p,
            "var_ratio_imp_over_obs": var_ratio,
            "iqr_ratio_imp_over_obs": iqr_ratio,
            "cliffs_delta": cd
        })

    return pd.DataFrame(rows).sort_values("KS_p")

# Example usage:
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
diag_df = pmm_diagnostic_tests(X_train, X_train_pmm, numeric_cols)

diag_df.head(20)

In [ ]:
def interpret_imputation_diagnostics(df, alpha=0.05):
    out = df.copy()

    # Statistical difference flags (p < alpha)
    out["KS_diff?"]   = out["KS_p"].apply(lambda p: "Yes" if (pd.notna(p) and p < alpha) else "No")
    out["AD_diff?"]   = out["AD_p"].apply(lambda p: "Yes" if (pd.notna(p) and p < alpha) else "No")
    out["MWU_shift?"] = out["MWU_p"].apply(lambda p: "Yes" if (pd.notna(p) and p < alpha) else "No")
    out["Var_diff?"]  = out["BrownForsythe_p"].apply(lambda p: "Yes" if (pd.notna(p) and p < alpha) else "No")

    # Practical spread checks (more important than p-values with large n)
    def spread_ok(r):
        if pd.isna(r):
            return "Unknown"
        return "OK" if 0.8 <= r <= 1.25 else "Not OK"

    out["Var_ratio_ok?"] = out["var_ratio_imp_over_obs"].apply(spread_ok)
    out["IQR_ratio_ok?"] = out["iqr_ratio_imp_over_obs"].apply(spread_ok)

    # Cliff's delta interpretation
    def effect_size_label(cd):
        if pd.isna(cd):
            return "Unknown"
        a = abs(cd)
        if a < 0.147:
            return "Negligible"
        elif a < 0.33:
            return "Small"
        elif a < 0.474:
            return "Medium"
        else:
            return "Large"

    out["Effect_size"] = out["cliffs_delta"].apply(effect_size_label)

    # Feature-level conclusion
    def feature_conclusion(row):
        dist_diff = (row["KS_diff?"] == "Yes") or (row["AD_diff?"] == "Yes")
        spread_bad = (row["Var_ratio_ok?"] == "Not OK") or (row["IQR_ratio_ok?"] == "Not OK")
        location_shift = (row["MWU_shift?"] == "Yes")

        if dist_diff and spread_bad:
            return "Poor (distribution + variability changed)"
        if dist_diff and not spread_bad:
            return "Moderate (distribution differs)"
        if (not dist_diff) and spread_bad:
            return "Moderate (variability mismatch)"
        if location_shift and dist_diff:
            return "Poor (shift + distribution differs)"
        return "Good / Acceptable"

    out["Conclusion"] = out.apply(feature_conclusion, axis=1)

    # Order + rounding for readability
    cols_order = [
        "feature", "n_obs", "n_imputed",
        "KS_p", "AD_p", "MWU_p", "BrownForsythe_p",
        "var_ratio_imp_over_obs", "iqr_ratio_imp_over_obs",
        "cliffs_delta", "Effect_size",
        "KS_diff?", "AD_diff?", "MWU_shift?", "Var_diff?",
        "Var_ratio_ok?", "IQR_ratio_ok?",
        "Conclusion"
    ]
    out = out[cols_order].copy()

    round_cols = [
        "KS_p", "AD_p", "MWU_p", "BrownForsythe_p",
        "var_ratio_imp_over_obs", "iqr_ratio_imp_over_obs",
        "cliffs_delta"
    ]
    out[round_cols] = out[round_cols].round(4)

    # Sort: worst first, then by KS_p
    out = out.sort_values(["Conclusion", "KS_p"], ascending=[True, True])

    return out

In [ ]:
import matplotlib.pyplot as plt

def plot_obs_vs_imputed(X_before, X_after, col, bins=30):
    miss = X_before[col].isna()
    obs = X_before.loc[~miss, col].astype(float)
    imp = X_after.loc[miss, col].astype(float)

    plt.figure(figsize=(7,4))
    plt.hist(obs, bins=bins, alpha=0.6, label="Observed")
    plt.hist(imp, bins=bins, alpha=0.6, label="Imputed (missing rows)")
    plt.title(f"{col}: Observed vs Imputed distribution")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.legend()
    plt.show()

for col in ["sleep_hours", "weekly_offline_hours", "daily_social_media_time"]:
    if col in X_train.columns:
        plot_obs_vs_imputed(X_train, X_train_pmm, col)

In [ ]:
features_imputed_pmm = [
    c for c in X_train.columns
    if X_train[c].isna().sum() > 0
]

print(features_imputed_pmm)

In [ ]:
from scipy.stats import ks_2samp
import pandas as pd

ks_rows = []

for col in features_imputed_pmm:
    before = X_train[col].dropna().astype(float)
    after  = X_train_pmm[col].astype(float)

    stat, p = ks_2samp(before, after, alternative="two-sided", mode="auto")

    ks_rows.append({
        "Feature": col,
        "KS Statistic": round(stat, 4),
        "p-value": round(p, 4),
        "Interpretation": "Distribution changed" if p < 0.05 else "No significant distribution change"
    })

ks_table_pmm = pd.DataFrame(ks_rows)
ks_table_pmm

In [ ]:
from scipy.stats import anderson_ksamp
import numpy as np

ad_rows = []

for col in features_imputed_pmm:
    before = X_train[col].dropna().astype(float).to_numpy()
    after  = X_train_pmm[col].astype(float).to_numpy()

    try:
        res = anderson_ksamp([before, after])
        stat = res.statistic
        p = res.significance_level / 100  # convert % -> proportion
    except Exception:
        stat = np.nan
        p = np.nan

    ad_rows.append({
        "Feature": col,
        "AD Statistic": round(stat, 4) if pd.notna(stat) else np.nan,
        "p-value": round(p, 4) if pd.notna(p) else np.nan,
        "Interpretation": "Distribution changed" if (pd.notna(p) and p < 0.05) else "No significant distribution change"
    })

ad_table_pmm = pd.DataFrame(ad_rows)
ad_table_pmm

In [ ]:
from scipy.stats import mannwhitneyu

mw_rows = []

for col in features_imputed_pmm:
    before = X_train[col].dropna().astype(float)
    after  = X_train_pmm[col].astype(float)

    stat, p = mannwhitneyu(before, after, alternative="two-sided")

    mw_rows.append({
        "Feature": col,
        "MWU Statistic": round(stat, 4),
        "p-value": round(p, 4),
        "Interpretation": "Median changed" if p < 0.05 else "Median preserved"
    })

mw_table_pmm = pd.DataFrame(mw_rows)
mw_table_pmm

In [ ]:
from scipy.stats import levene

bf_rows = []

for col in features_imputed_pmm:
    before = X_train[col].dropna().astype(float)
    after  = X_train_pmm[col].astype(float)

    stat, p = levene(before, after, center="median")

    bf_rows.append({
        "Feature": col,
        "BF Statistic": round(stat, 4),
        "p-value": round(p, 4),
        "Interpretation": "Variance changed" if p < 0.05 else "Variance preserved"
    })

bf_table_pmm = pd.DataFrame(bf_rows)
bf_table_pmm

In [ ]:
import numpy as np

def cliffs_delta_fast(x, y, max_n=4000, seed=42):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[~np.isnan(x)]
    y = y[~np.isnan(y)]

    if len(x) == 0 or len(y) == 0:
        return np.nan

    rng = np.random.RandomState(seed)
    if len(x) > max_n:
        x = rng.choice(x, size=max_n, replace=False)
    if len(y) > max_n:
        y = rng.choice(y, size=max_n, replace=False)

    gt = (x[:, None] > y[None, :]).mean()
    lt = (x[:, None] < y[None, :]).mean()
    return gt - lt

cd_rows = []

for col in features_imputed_pmm:
    before = X_train[col].dropna().astype(float).to_numpy()
    after  = X_train_pmm[col].astype(float).to_numpy()

    delta = cliffs_delta_fast(before, after)

    abs_d = abs(delta)

    if abs_d < 0.147:
        interpretation = "Negligible difference"
    elif abs_d < 0.33:
        interpretation = "Small difference"
    elif abs_d < 0.474:
        interpretation = "Medium difference"
    else:
        interpretation = "Large difference"

    cd_rows.append({
        "Feature": col,
        "Cliff's Delta": round(delta, 4),
        "Interpretation": interpretation
    })

cd_table_pmm = pd.DataFrame(cd_rows)
cd_table_pmm

In [ ]:
pd.set_option("display.max_colwidth", None)

summary_rows = []

total_features = len(ks_table_pmm)


# Kolmogorov–Smirnov
ks_sig = (ks_table_pmm["p-value"] < 0.05).sum()

summary_rows.append({
    "Test": "Kolmogorov–Smirnov",
    "Features tested": total_features,
    "Significant differences": ks_sig,
    "Interpretation":
        "Distribution preserved"
        if ks_sig == 0 else
        "Some distribution differences"
})


# Anderson–Darling
ad_sig = (ad_table_pmm["p-value"] < 0.05).sum()

summary_rows.append({
    "Test": "Anderson–Darling",
    "Features tested": total_features,
    "Significant differences": ad_sig,
    "Interpretation":
        "Highly sensitive; effect size should be considered"
})


# Mann–Whitney U
mw_sig = (mw_table_pmm["p-value"] < 0.05).sum()

summary_rows.append({
    "Test": "Mann–Whitney U",
    "Features tested": total_features,
    "Significant differences": mw_sig,
    "Interpretation":
        "Median preserved"
        if mw_sig == 0 else
        "Median changed in some features"
})


# Brown–Forsythe
bf_sig = (bf_table_pmm["p-value"] < 0.05).sum()

summary_rows.append({
    "Test": "Brown–Forsythe",
    "Features tested": total_features,
    "Significant differences": bf_sig,
    "Interpretation":
        "Variance preserved"
        if bf_sig == 0 else
        "Variance changed in some features"
})


# Cliff's Delta
cd_sig = (cd_table_pmm["Interpretation"] != "Negligible difference").sum()

summary_rows.append({
    "Test": "Cliff's Delta",
    "Features tested": total_features,
    "Significant differences": cd_sig,
    "Interpretation":
        "No meaningful practical difference"
        if cd_sig == 0 else
        "Practical difference exists"
})


summary_table_pmm = pd.DataFrame(summary_rows)

summary_table_pmm

#Finalize the dataset

In [ ]:
X_tr = X_train_pmm.copy()
X_ts = X_test_pmm.copy()

In [ ]:
y_tr = y_train.copy()
y_ts = y_test.copy()

In [ ]:
df_tr = pd.concat([X_tr, y_tr], axis=1)
df_ts = pd.concat([X_ts, y_ts], axis=1)

In [ ]:
df_tr.info()

#Cluster Analysis

##K - Prototype

In [ ]:
# Install if not already
!pip install kmodes

# Imports
import numpy as np
import pandas as pd

from sklearn.preprocessing import RobustScaler
from kmodes.kprototypes import KPrototypes

import matplotlib.pyplot as plt

In [ ]:
# Target (only for analysis, NOT clustering)
target_col = "actual_productivity_score"


# CATEGORICAL
cat_cols = [
    "gender",
    "job_type",
    "social_platform_preference",
    "stress_level",
]


# BINARY → treat as categorical
bin_cols = [
    "uses_focus_apps",
    "has_digital_wellbeing_enabled",
]


# NUMERIC
num_cols = [
    "age",
    "daily_social_media_time",
    "number_of_notifications",
    "work_hours_per_day",
    "sleep_hours",
    "screen_time_before_sleep",
    "breaks_during_work",
    "coffee_consumption_per_day",
    "days_feeling_burnout_per_month",
    "weekly_offline_hours",
    "job_satisfaction_score",
]


# All clustering columns
all_cols = num_cols + cat_cols + bin_cols

In [ ]:
df_kp = df_tr.copy()

# Keep required columns
df_kp = df_kp[all_cols + [target_col]].copy()


# Convert categorical → string
for c in cat_cols:
    df_kp[c] = df_kp[c].astype(str)


# Convert binary → string
for b in bin_cols:
    df_kp[b] = df_kp[b].astype(int).astype(str)


# Numeric → float
for n in num_cols:
    df_kp[n] = df_kp[n].astype(float)


# Drop missing
df_kp = df_kp.dropna().reset_index(drop=True)


df_kp.head()

In [ ]:
scaler = RobustScaler()

X_num_scaled = scaler.fit_transform(df_kp[num_cols])

X_num_scaled = pd.DataFrame(
    X_num_scaled,
    columns=num_cols,
    index=df_kp.index
)

In [ ]:
X_mix = pd.concat(
    [X_num_scaled, df_kp[cat_cols + bin_cols]],
    axis=1
)


X_mat = X_mix.to_numpy()


# Get categorical indices
cat_idx = [
    X_mix.columns.get_loc(c)
    for c in cat_cols + bin_cols
]


print("Shape:", X_mix.shape)
print("Categorical indices:", cat_idx)

In [ ]:
costs = []

K_range = range(2, 11)

for k in K_range:

    kp = KPrototypes(
        n_clusters=k,
        init="Cao",
        n_init=5,
        random_state=42
    )

    labels = kp.fit_predict(
        X_mat,
        categorical=cat_idx
    )

    costs.append(kp.cost_)

    print("k =", k, " cost =", kp.cost_)

In [ ]:
plt.figure(figsize=(7,4))

plt.plot(K_range, costs, marker="o")

plt.xlabel("k")

plt.ylabel("Cost")

plt.title("Elbow Method")

plt.grid()

plt.show()

In [ ]:
best_k = 5

In [ ]:
final_kp = KPrototypes(

    n_clusters=best_k,

    init="Cao",

    n_init=10,

    random_state=42
)


clusters = final_kp.fit_predict(

    X_mat,

    categorical=cat_idx

)


df_clustered = df_kp.copy()

df_clustered["cluster"] = clusters


df_clustered.head()

In [ ]:
df_clustered["cluster"].value_counts()

In [ ]:
df_clustered.groupby("cluster")[num_cols].mean()

In [ ]:
df_clustered.groupby("cluster")[num_cols].mean()

In [ ]:
df_clustered.groupby("cluster")[cat_cols + bin_cols].agg(

    lambda x: x.mode()[0]

)

In [ ]:
df_clustered.groupby("cluster")[target_col].mean().sort_values(ascending=False)

In [ ]:
from scipy.stats import kruskal

groups = [
    df_clustered[df_clustered["cluster"] == c]["actual_productivity_score"]
    for c in sorted(df_clustered["cluster"].unique())
]

stat, p = kruskal(*groups)

print("Kruskal-Wallis H:", stat)
print("p-value:", p)

In [ ]:
from scipy.stats import mannwhitneyu
import itertools

clusters = sorted(df_clustered["cluster"].unique())

for c1, c2 in itertools.combinations(clusters, 2):

    g1 = df_clustered[df_clustered.cluster==c1]["actual_productivity_score"]
    g2 = df_clustered[df_clustered.cluster==c2]["actual_productivity_score"]

    stat, p = mannwhitneyu(g1, g2)

    print(c1, "vs", c2, "p =", p)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

sns.boxplot(
    x="cluster",
    y="actual_productivity_score",
    data=df_clustered
)

plt.title("Productivity by Cluster")

plt.show()

##DBSCAN

In [ ]:
import numpy as np
import pandas as pd

# For FAMD
from prince import FAMD

# For clustering and evaluation
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

# Optional plotting
import matplotlib.pyplot as plt

# Optional scaling for numeric variables (before FAMD)
from sklearn.preprocessing import RobustScaler


In [ ]:
X_train_dbs = X_tr.copy()
X_test_dbs = X_ts.copy()


**Define Column Groups**

In [ ]:
numeric_cols = X_train_dbs.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X_train_dbs.select_dtypes(include=["object", "bool"]).columns.tolist()

**Preprocessing with Robust Scaling**

In [ ]:
# Ensure categorical columns are of type 'category' (required by FAMD)
for col in categorical_cols:
    X_train_dbs[col] = X_train_dbs[col].astype("category")

# Optional: scale numeric columns if desired (FAMD also standardizes internally)
# from sklearn.preprocessing import RobustScaler
# scaler = RobustScaler()
# X_train_dbs[numeric_cols] = scaler.fit_transform(X_train_dbs[numeric_cols])


**FAMD**

In [ ]:
import prince

# Initialize FAMD
famd = prince.FAMD(
    n_components=6,
    n_iter=3,        # default is 3; more iterations can improve stability
    copy=True,
    check_input=True,
    random_state=42
)

# Fit FAMD on the mixed-type dataset
famd = famd.fit(X_train_dbs)

# Transform the data to numeric embeddings for DBSCAN
X_train_processed = famd.transform(X_train_dbs)


**Step 6 — Find Good eps Using k-distance Graph**

In [ ]:
neighbors = NearestNeighbors(n_neighbors=5)
neighbors_fit = neighbors.fit(X_train_processed)
distances, indices = neighbors_fit.kneighbors(X_train_processed)

distances = np.sort(distances[:, 4])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(distances)
plt.title("K-distance Graph (k=5) - FAMD Embedding")
plt.xlabel("Data Points sorted by distance")
plt.ylabel("5th Nearest Neighbor Distance")
plt.show()


In [ ]:
eps_values = [4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 8.5, 9, 9.5, 10, 10.5]

for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5)
    clusters = db.fit_predict(X_train_processed)

    n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
    n_noise = list(clusters).count(-1)

    print(f"eps={eps}")
    print("Clusters:", n_clusters)
    print("Noise points:", n_noise)

    if n_clusters > 1:
        try:
            score = silhouette_score(X_train_processed, clusters)
            print("Silhouette:", score)
        except ValueError:
            print("Silhouette cannot be computed (e.g., all noise or single cluster)")

    print("-" * 40)


**Step 7 — Apply DBSCAN**

In [ ]:
db = DBSCAN(eps=7, min_samples=5)
cluster_labels = db.fit_predict(X_train_processed)

# Add cluster labels to original data (for analysis)
X_train_dbs_with_labels = X_train_dbs.copy()
X_train_dbs_with_labels['cluster'] = cluster_labels



**Step 8 — Evaluate Clustering**

In [ ]:
n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
n_noise = list(clusters).count(-1)

print("Number of clusters:", n_clusters)
print("Number of noise points:", n_noise)


In [ ]:
if n_clusters > 1:
    score = silhouette_score(X_train_processed, clusters)
    print("Silhouette Score:", score)


**Step 9 — Interpret Clusters**

In [ ]:
# Numeric features summary
numeric_summary = X_train_dbs_with_labels.groupby('cluster')[numeric_cols].mean()
print(numeric_summary)

import numpy as np
import pandas as pd

def safe_mode(s: pd.Series):
    s = s.dropna()
    if s.empty:
        return np.nan
    return s.mode().iloc[0]

categorical_summary = (
    X_train_dbs_with_labels
    .groupby("cluster")[categorical_cols]
    .agg(safe_mode)
)

categorical_summary


In [ ]:
y_train_with_labels = y_train.copy()
y_train_with_labels = pd.DataFrame(y_train_with_labels)
y_train_with_labels['cluster'] = cluster_labels

productivity_summary = y_train_with_labels.groupby('cluster')['actual_productivity_score'].mean()
print(productivity_summary)


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.scatter(
    X_train_processed[0],
    X_train_processed[1],
    c=cluster_labels,
    cmap="Set1",
    alpha=0.6
)
plt.title("Components 1 & 2")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.colorbar(label="Cluster")

plt.subplot(1, 3, 2)
plt.scatter(
    X_train_processed[2],
    X_train_processed[3],
    c=cluster_labels,
    cmap="Set1",
    alpha=0.6
)
plt.title("Components 3 & 4")
plt.xlabel("Component 3")
plt.ylabel("Component 4")
plt.colorbar(label="Cluster")

plt.subplot(1, 3, 3)
plt.scatter(
    X_train_processed[4],
    X_train_processed[5],
    c=cluster_labels,
    cmap="Set1",
    alpha=0.6
)
plt.title("Components 5 & 6")
plt.xlabel("Component 5")
plt.ylabel("Component 6")
plt.colorbar(label="Cluster")

plt.suptitle("DBSCAN Clusters (FAMD-reduced)", y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

sns.boxplot(x=cluster_labels, y=y_train)
plt.title("Productivity Distribution by DBSCAN Cluster")
plt.show()

In [ ]:
# Copy original training predictors
X_train_out = X_tr.copy()

# Add cluster labels from DBSCAN
X_train_out["cluster"] = cluster_labels

# Add productivity target
X_train_out["actual_productivity_score"] = y_tr.values

In [ ]:
outliers = X_train_out[X_train_out["cluster"] == -1]

print("Number of outliers:", len(outliers))
outliers.head()

In [ ]:
normal = X_train_out[X_train_out["cluster"] != -1]

comparison = pd.DataFrame({
    "Outlier Mean": outliers.mean(numeric_only=True),
    "Normal Mean": normal.mean(numeric_only=True)
})

comparison["Difference"] = comparison["Outlier Mean"] - comparison["Normal Mean"]

comparison.sort_values("Difference", key=abs, ascending=False)

In [ ]:
features_to_check = [
    "daily_social_media_time",
    "work_hours_per_day",
    "stress_level",
    "sleep_hours",
    "job_satisfaction_score"
]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))

sns.boxplot(
    data=X_train_out,
    x="cluster",
    y="actual_productivity_score"
)

plt.title("Productivity Distribution: Outliers vs Normal")
plt.show()

In [ ]:
print(X_train_out[features_to_check].dtypes)

In [ ]:
for col in features_to_check:

    normal[col] = pd.to_numeric(normal[col], errors='coerce')
    outliers[col] = pd.to_numeric(outliers[col], errors='coerce')

In [ ]:
outliers.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

for col in features_to_check:

    normal_vals = pd.to_numeric(normal[col], errors='coerce')
    outlier_vals = pd.to_numeric(outliers[col], errors='coerce')

    plt.figure(figsize=(6,4))

    sns.kdeplot(x=normal_vals, label="Normal", fill=True)
    sns.kdeplot(x=outlier_vals, label="Outliers", fill=True)

    plt.title(f"{col} Distribution")
    plt.legend()

    plt.show()

#EDA

For EDA use df which is a copy of df_train_fe

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df_tr, title="YData Profiling Report for df_tr", explorative = True)
print("Profile report object created.")

In [ ]:
profile.to_file("ydata_profiling_report_df_tr.html")
print("YData Profiling Report saved to ydata_profiling_report_df_tr.html")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import LabelEncoder


def get_mi_scores(X, y):

    X_copy = X.copy()

    # -----------------------------
    # 1. Handle categorical columns
    # -----------------------------
    label_encoders = {}

    for col in X_copy.select_dtypes(include=['object', 'category']).columns:
        le = LabelEncoder()
        X_copy[col] = le.fit_transform(X_copy[col].astype(str))
        label_encoders[col] = le

    # -----------------------------
    # 2. Convert boolean → int
    # -----------------------------
    for col in X_copy.select_dtypes(include=['bool']).columns:
        X_copy[col] = X_copy[col].astype(int)

    # -----------------------------
    # 3. Handle missing values
    # -----------------------------
    X_copy = X_copy.fillna(X_copy.median(numeric_only=True))

    # -----------------------------
    # 4. Identify discrete features
    # -----------------------------
    discrete_features = []

    for col in X_copy.columns:
        if (
            pd.api.types.is_integer_dtype(X_copy[col]) or
            X_copy[col].nunique() < 15
        ):
            discrete_features.append(True)
        else:
            discrete_features.append(False)

    # -----------------------------
    # 5. Compute Mutual Information
    # -----------------------------
    print("Calculating MI scores...")

    mi_scores_array = mutual_info_regression(
        X_copy,
        y,
        discrete_features=discrete_features,
        random_state=42,
        n_neighbors=5   # more stable estimation
    )

    mi_scores = pd.Series(
        mi_scores_array,
        index=X_copy.columns,
        name="MI Score"
    ).sort_values(ascending=False)

    return mi_scores


# ======================================
# Execute MI calculation
# ======================================

columns_to_exclude = [
    'extreme_social_media_user',
    'screen_time_before_sleep_binned',
    'work_hours_per_day_binned'
]

X_tr_filtered = X_tr.drop(columns=columns_to_exclude, errors='ignore')

mi_scores = get_mi_scores(X_tr_filtered, y_tr)

print("\n--- Mutual Information Scores ---")
print(mi_scores)


# ======================================
# Plot
# ======================================

plt.figure(figsize=(10, 8))

sns.barplot(
    x=mi_scores.values,
    y=mi_scores.index,
    palette="magma"
)

plt.title("Feature Importance (Mutual Information)", fontsize=14)
plt.xlabel("Mutual Information Score")
plt.ylabel("Features")

plt.tight_layout()
plt.show()

##Correlations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Identify all numerical columns from df_tr
numerical_cols = df_tr.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove 'actual_productivity_score' if it's present, as it's the target
if 'actual_productivity_score' in numerical_cols:
    numerical_cols.remove('actual_productivity_score')

# Ensure 'job_satisfaction_score' is included for the correlation
if 'job_satisfaction_score' not in numerical_cols:
    numerical_cols.append('job_satisfaction_score')

# Calculate the correlation matrix for the subset of numerical columns
corr_matrix_satisfaction = df_tr[numerical_cols].corr()[['job_satisfaction_score']].sort_values(by='job_satisfaction_score', ascending=False)

# Plot the heatmap
plt.figure(figsize=(8, 10))
sns.heatmap(corr_matrix_satisfaction, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation with Job Satisfaction Score", fontsize=16)
plt.yticks(rotation=0)
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Identify all numerical columns from df_tr
numerical_cols_for_spearman = df_tr.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove 'actual_productivity_score' if it's present, as it's the target
if 'actual_productivity_score' in numerical_cols_for_spearman:
    numerical_cols_for_spearman.remove('actual_productivity_score')

# Ensure 'job_satisfaction_score' is included for the correlation
if 'job_satisfaction_score' not in numerical_cols_for_spearman:
    numerical_cols_for_spearman.append('job_satisfaction_score')

# Calculate Spearman's correlation matrix for the subset of numerical columns
spearman_corr_matrix_satisfaction = df_tr[numerical_cols_for_spearman].corr(method='spearman')[['job_satisfaction_score']].sort_values(by='job_satisfaction_score', ascending=False)

# Plot the heatmap
plt.figure(figsize=(8, 10))
sns.heatmap(spearman_corr_matrix_satisfaction, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Spearman's Correlation with Job Satisfaction Score", fontsize=16)
plt.yticks(rotation=0)
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

display(spearman_corr_matrix_satisfaction)

In [ ]:
from sklearn.feature_selection import mutual_info_regression
import pandas as pd

# Use only predictors (exclude squared feature if derived)
predictors = X_tr.copy()

# convert categoricals to codes
for col in predictors.select_dtypes(['category']).columns:
    predictors[col] = predictors[col].cat.codes

mi_matrix = pd.DataFrame(index=predictors.columns, columns=predictors.columns)

for target in predictors.columns:

    X_other = predictors.drop(columns=[target])
    y_target = predictors[target]

    mi = mutual_info_regression(X_other, y_target, random_state=42)

    mi_matrix.loc[target, X_other.columns] = mi

mi_matrix = mi_matrix.astype(float)

mi_matrix.mean().sort_values(ascending=False)

In [ ]:
# Calculate average MI score per variable
mi_scores = mi_matrix.mean()

# Convert to proper table
mi_table = (
    mi_scores
    .reset_index()
    .rename(columns={
        "index": "Variable",
        0: "Average Mutual Information Score"
    })
    .sort_values("Average Mutual Information Score", ascending=False)
)

mi_table

In [ ]:
def interpret_mi(score):
    if score < 0.005:
        return "Very Weak"
    elif score < 0.02:
        return "Weak"
    elif score < 0.1:
        return "Moderate"
    else:
        return "Strong"

mi_table["Relationship Strength"] = mi_table["Average Mutual Information Score"].apply(interpret_mi)

mi_table

In [ ]:
# ======================================
# Set target and predictors
# ======================================

target = "job_satisfaction_score"

columns_to_exclude = [
    "job_satisfaction_score",          # target itself
    "actual_productivity_score",      # exclude productivity
    "perceived_productivity_score",   # exclude productivity

    # optional: your engineered features if needed
    'extreme_social_media_user',
    'screen_time_before_sleep_binned',
    'work_hours_per_day_binned'
]

# Define X and y
X_js = X_tr.drop(columns=columns_to_exclude, errors="ignore")
y_js = X_tr[target]


# ======================================
# Calculate MI scores
# ======================================

mi_scores_js = get_mi_scores(X_js, y_js)

print("\n--- Mutual Information Scores (Target: Job Satisfaction) ---")
print(mi_scores_js)


# ======================================
# Plot
# ======================================

plt.figure(figsize=(10, 8))

sns.barplot(
    x=mi_scores_js.values,
    y=mi_scores_js.index,
    palette="magma"
)

plt.title("Feature Importance for Job Satisfaction (Mutual Information)", fontsize=14)
plt.xlabel("Mutual Information Score")
plt.ylabel("Features")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

predictors = X_tr.copy()

for col in predictors.select_dtypes(['category']).columns:
    predictors[col] = predictors[col].cat.codes


results = {}

for target in predictors.columns:

    X_other = predictors.drop(columns=[target])
    y_target = predictors[target]

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_other, y_target)

    pred = model.predict(X_other)

    results[target] = r2_score(y_target, pred)

pd.Series(results).sort_values()

In [ ]:
import scipy.stats as stats
import numpy as np

def cramers_v(x, y):
    confusion = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(confusion)[0]
    n = confusion.sum().sum()
    phi2 = chi2/n
    r,k = confusion.shape
    return np.sqrt(phi2/min(k-1, r-1))

cramers_v(X_tr['gender'], X_tr['job_type'])

##Bivariate

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
import numpy as np

# Create the scatter plot as before
sns.scatterplot(x=X_tr["job_satisfaction_score"], y=y_tr)

# Fit a simple linear regression model
linear_model = LinearRegression()
X_feature = X_tr["job_satisfaction_score"].values.reshape(-1, 1)
y_target = y_tr.values

linear_model.fit(X_feature, y_target)

# Predict values using the fitted model
y_pred_line = linear_model.predict(X_feature)

# Plot the regression line
plt.plot(X_tr["job_satisfaction_score"], y_pred_line, color='red', linestyle='--', label='Linear Regression Line')

plt.title('Job Satisfaction Score vs. Actual Productivity Score with Linear Regression')
plt.xlabel('Job Satisfaction Score')
plt.ylabel('Actual Productivity Score')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
import numpy as np
from scipy import stats

# Fit a simple linear regression model
linear_model = LinearRegression()
X_feature = X_tr["job_satisfaction_score"].values.reshape(-1, 1)
y_target = y_tr.values

linear_model.fit(X_feature, y_target)

# Predict values using the fitted model
y_pred_line = linear_model.predict(X_feature)

# --- Residual Analysis ---

# Calculate residuals
residuals = y_target - y_pred_line

# Create a single canvas for all residual plots
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Plot 1: Residuals vs. Fitted values
sns.scatterplot(x=y_pred_line, y=residuals, ax=axes[0], alpha=0.6)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title('Residuals vs. Fitted Values')
axes[0].set_xlabel('Fitted Values (Predicted Productivity)')
axes[0].set_ylabel('Residuals')

# Plot 2: Histogram of Residuals
sns.histplot(residuals, kde=True, ax=axes[1])
axes[1].set_title('Histogram of Residuals')
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')

# Plot 3: QQ Plot of Residuals
stats.probplot(residuals, dist="norm", plot=axes[2])
axes[2].set_title('QQ Plot of Residuals')

plt.tight_layout()
plt.show()

##screen time before sleep binned

In [ ]:
# -------------------------------
# 5️⃣ Binning 'screen_time_before_sleep'
# -------------------------------
bins = [-np.inf, 1, 2, 3]
labels = ['<=1 hour', '1-2 hours', '2-3 hours']
df_tr['screen_time_before_sleep_binned'] = pd.cut(df_tr['screen_time_before_sleep'], bins=bins, labels=labels, right=True)


In [ ]:
# Ensure df_tr has the binned feature by applying the feature engineering function
# The add_new_features function was defined previously in cell `gdCyEz2aco6t`
# Make sure df_tr is a fresh copy of df_train before applying features to avoid re-applying

import matplotlib.pyplot as plt
import seaborn as sns

# Get unique categories for iteration
categories = df_tr['screen_time_before_sleep_binned'].unique()
categories = sorted(categories, key=lambda x: str(x))

n_categories = len(categories)
n_cols = 2  # Number of columns for subplots
n_rows = (n_categories + n_cols - 1) // n_cols # Calculate number of rows needed

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten()

for i, category in enumerate(categories):
    subset = df_tr[df_tr['screen_time_before_sleep_binned'] == category]
    sns.histplot(subset['actual_productivity_score'], kde=True, bins=20, ax=axes[i], color='skyblue')
    axes[i].set_title(f'Productivity Distribution for {category}')
    axes[i].set_xlabel('Actual Productivity Score')
    axes[i].set_ylabel('Frequency')

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from scipy import stats

# Get unique categories for iteration
categories = df_tr['screen_time_before_sleep_binned'].unique()
categories = sorted(categories, key=lambda x: str(x))

# --- ANOVA Test ---

# Prepare data for ANOVA
# Get the actual_productivity_score for each category
groups = []
for category in categories:
    group_data = df_tr[df_tr['screen_time_before_sleep_binned'] == category]['actual_productivity_score']
    groups.append(group_data)

# Perform one-way ANOVA
f_statistic, p_value = stats.f_oneway(*groups)

print("\n--- ANOVA Test Results ---")
print(f"F-statistic: {f_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Conclusion: The p-value is less than 0.05, indicating a statistically significant difference in actual productivity scores among the screen time before sleep categories.")
else:
    print("Conclusion: The p-value is not less than 0.05, indicating no statistically significant difference in actual productivity scores among the screen time before sleep categories.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations

# Ensure df_tr is used directly as it already has feature engineered columns
# No new copies of df_train or re-application of add_new_features

# Get unique categories for iteration (from the existing df_tr)
categories = df_tr['screen_time_before_sleep_binned'].unique()
categories = sorted(categories, key=lambda x: str(x))

# --- Kolmogorov-Smirnov (KS) Test for pairs of categories ---
print("\n--- Kolmogorov-Smirnov (KS) Test Results for Productivity Distributions ---")

# Iterate through all unique pairs of categories
for cat1, cat2 in combinations(categories, 2):
    group1 = df_tr[df_tr['screen_time_before_sleep_binned'] == cat1]['actual_productivity_score'].dropna()
    group2 = df_tr[df_tr['screen_time_before_sleep_binned'] == cat2]['actual_productivity_score'].dropna()

    # Check if groups have enough data for KS test (at least 2 observations)
    if len(group1) > 1 and len(group2) > 1:
        ks_statistic, p_value = stats.ks_2samp(group1, group2)

        print(f"\nComparing '{cat1}' vs '{cat2}':")
        print(f"  KS Statistic = {ks_statistic:.4f}")
        print(f"  P-value      = {p_value:.4f}")

        if p_value < 0.05:
            print("  Conclusion   = Statistically significant difference in productivity distributions (p < 0.05)")
        else:
            print("  Conclusion   = No statistically significant difference in productivity distributions (p >= 0.05)")
    else:
        print(f"\nSkipping comparison '{cat1}' vs '{cat2}' due to insufficient data.")

In [ ]:
X_tr = df_tr.drop(columns=["actual_productivity_score"])
y_tr = df_tr["actual_productivity_score"]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import LabelEncoder


def get_mi_scores(X, y):

    X_copy = X.copy()

    # -----------------------------
    # 1. Handle categorical columns
    # -----------------------------
    label_encoders = {}

    for col in X_copy.select_dtypes(include=['object', 'category']).columns:
        le = LabelEncoder()
        X_copy[col] = le.fit_transform(X_copy[col].astype(str))
        label_encoders[col] = le

    # -----------------------------
    # 2. Convert boolean → int
    # -----------------------------
    for col in X_copy.select_dtypes(include=['bool']).columns:
        X_copy[col] = X_copy[col].astype(int)

    # -----------------------------
    # 3. Handle missing values
    # -----------------------------
    X_copy = X_copy.fillna(X_copy.median(numeric_only=True))

    # -----------------------------
    # 4. Identify discrete features
    # -----------------------------
    discrete_features = []

    for col in X_copy.columns:
        if (
            pd.api.types.is_integer_dtype(X_copy[col]) or
            X_copy[col].nunique() < 15
        ):
            discrete_features.append(True)
        else:
            discrete_features.append(False)

    # -----------------------------
    # 5. Compute Mutual Information
    # -----------------------------
    print("Calculating MI scores...")

    mi_scores_array = mutual_info_regression(
        X_copy,
        y,
        discrete_features=discrete_features,
        random_state=42,
        n_neighbors=5   # more stable estimation
    )

    mi_scores = pd.Series(
        mi_scores_array,
        index=X_copy.columns,
        name="MI Score"
    ).sort_values(ascending=False)

    return mi_scores


# ======================================
# Execute MI calculation
# ======================================

columns_to_exclude = [
    'screen_time_before_sleep_binned'
]

X_tr_filtered = X_tr.drop(columns=columns_to_exclude, errors='ignore')

mi_scores = get_mi_scores(X_tr_filtered, y_tr)

print("\n--- Mutual Information Scores ---")
print(mi_scores)


# ======================================
# Plot
# ======================================

plt.figure(figsize=(10, 8))

sns.barplot(
    x=mi_scores.values,
    y=mi_scores.index,
    palette="magma"
)

plt.title("Feature Importance (Mutual Information)", fontsize=14)
plt.xlabel("Mutual Information Score")
plt.ylabel("Features")

plt.tight_layout()
plt.show()

In [ ]:
df_tr =df_tr.drop(columns=['screen_time_before_sleep_binned'])

##work hours per day binned

In [ ]:
# -------------------------------
# 6️⃣ Binning 'work_hours_per_day'
# -------------------------------
bins_work_hours = [-0.1, 6.0, 8.0, 12.0]
labels_work_hours = ['student/Part-time', 'Full-time', 'Overtime']
df_tr['work_hours_per_day_binned'] = pd.cut(df_tr['work_hours_per_day'], bins=bins_work_hours, labels=labels_work_hours, right=True)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

print("Current categories in 'work_hours_per_day_binned' in df_tr:")
display(df_tr['work_hours_per_day_binned'].value_counts(dropna=False))

# Get unique categories for iteration
categories_work = df_tr['work_hours_per_day_binned'].unique()
categories_work = sorted(categories_work, key=lambda x: str(x)) # Ensure consistent order

n_categories_work = len(categories_work)
n_cols_work = 2  # Number of columns for subplots
n_rows_work = (n_categories_work + n_cols_work - 1) // n_cols_work # Calculate number of rows needed

fig_work, axes_work = plt.subplots(n_rows_work, n_cols_work, figsize=(15, 5 * n_rows_work))
axes_work = axes_work.flatten()

for i, category in enumerate(categories_work):
    subset_work = df_tr[df_tr['work_hours_per_day_binned'] == category]
    sns.histplot(subset_work['actual_productivity_score'], kde=True, bins=20, ax=axes_work[i], color='lightcoral')
    axes_work[i].set_title(f'Productivity Distribution for {category}')
    axes_work[i].set_xlabel('Actual Productivity Score')
    axes_work[i].set_ylabel('Frequency')

# Hide any unused subplots
for j in range(i + 1, len(axes_work)):
    fig_work.delaxes(axes_work[j])

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations

# Get unique categories for iteration (from the existing df_tr)
categories_work = df_tr['work_hours_per_day_binned'].unique()
categories_work = sorted(categories_work, key=lambda x: str(x))

# --- ANOVA Test ---
print("\n--- ANOVA Test Results (work_hours_per_day_binned) ---")

# Prepare data for ANOVA
groups_anova = []
for category in categories_work:
    group_data = df_tr[df_tr['work_hours_per_day_binned'] == category]['actual_productivity_score'].dropna()
    if not group_data.empty: # Only add non-empty groups
        groups_anova.append(group_data)

if len(groups_anova) > 1:
    f_statistic, p_value_anova = stats.f_oneway(*groups_anova)

    print(f"F-statistic: {f_statistic:.4f}")
    print(f"P-value: {p_value_anova:.4f}")

    if p_value_anova < 0.05:
        print("Conclusion: The p-value is less than 0.05, indicating a statistically significant difference in actual productivity scores among these work hour categories.")
    else:
        print("Conclusion: The p-value is not less than 0.05, indicating no statistically significant difference in actual productivity scores among these work hour categories.")
else:
    print("Not enough groups with data to perform ANOVA test.")

# --- Kolmogorov-Smirnov (KS) Test for pairs of categories ---
print("\n--- Kolmogorov-Smirnov (KS) Test Results for Productivity Distributions (work_hours_per_day_binned) ---")

# Iterate through all unique pairs of categories
for cat1, cat2 in combinations(categories_work, 2):
    group1_ks = df_tr[df_tr['work_hours_per_day_binned'] == cat1]['actual_productivity_score'].dropna()
    group2_ks = df_tr[df_tr['work_hours_per_day_binned'] == cat2]['actual_productivity_score'].dropna()

    # Check if groups have enough data for KS test (at least 2 observations)
    if len(group1_ks) > 1 and len(group2_ks) > 1:
        ks_statistic, p_value_ks = stats.ks_2samp(group1_ks, group2_ks)

        print(f"\nComparing '{cat1}' vs '{cat2}':")
        print(f"  KS Statistic = {ks_statistic:.4f}")
        print(f"  P-value      = {p_value_ks:.4f}")

        if p_value_ks < 0.05:
            print("  Conclusion   = Statistically significant difference in productivity distributions (p < 0.05)")
        else:
            print("  Conclusion   = No statistically significant difference in productivity distributions (p >= 0.05)")
    else:
        print(f"\nSkipping comparison '{cat1}' vs '{cat2}' due to insufficient data.")

In [ ]:
# --- Kruskal–Wallis Test ---
print("\n--- Kruskal–Wallis Test Results (work_hours_per_day_binned) ---")

# Prepare data for Kruskal–Wallis
groups_kw = []

for category in categories_work:

    group_data = df_tr[
        df_tr['work_hours_per_day_binned'] == category
    ]['actual_productivity_score'].dropna()

    if not group_data.empty:   # only include non-empty groups
        groups_kw.append(group_data)


# Perform test only if enough groups exist
if len(groups_kw) > 1:

    h_statistic, p_value_kw = stats.kruskal(*groups_kw)

    print(f"H-statistic: {h_statistic:.4f}")
    print(f"P-value: {p_value_kw:.4f}")

    if p_value_kw < 0.05:

        print("Conclusion: The p-value is less than 0.05, indicating a statistically significant difference in productivity among work hour categories.")

    else:

        print("Conclusion: The p-value is not less than 0.05, indicating no statistically significant difference in productivity among work hour categories.")

else:

    print("Not enough groups with data to perform Kruskal–Wallis test.")

In [ ]:
X_tr = df_tr.drop(columns=["actual_productivity_score"])
y_tr = df_tr["actual_productivity_score"]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import LabelEncoder


def get_mi_scores(X, y):

    X_copy = X.copy()

    # -----------------------------
    # 1. Handle categorical columns
    # -----------------------------
    label_encoders = {}

    for col in X_copy.select_dtypes(include=['object', 'category']).columns:
        le = LabelEncoder()
        X_copy[col] = le.fit_transform(X_copy[col].astype(str))
        label_encoders[col] = le

    # -----------------------------
    # 2. Convert boolean → int
    # -----------------------------
    for col in X_copy.select_dtypes(include=['bool']).columns:
        X_copy[col] = X_copy[col].astype(int)

    # -----------------------------
    # 3. Handle missing values
    # -----------------------------
    X_copy = X_copy.fillna(X_copy.median(numeric_only=True))

    # -----------------------------
    # 4. Identify discrete features
    # -----------------------------
    discrete_features = []

    for col in X_copy.columns:
        if (
            pd.api.types.is_integer_dtype(X_copy[col]) or
            X_copy[col].nunique() < 15
        ):
            discrete_features.append(True)
        else:
            discrete_features.append(False)

    # -----------------------------
    # 5. Compute Mutual Information
    # -----------------------------
    print("Calculating MI scores...")

    mi_scores_array = mutual_info_regression(
        X_copy,
        y,
        discrete_features=discrete_features,
        random_state=42,
        n_neighbors=5   # more stable estimation
    )

    mi_scores = pd.Series(
        mi_scores_array,
        index=X_copy.columns,
        name="MI Score"
    ).sort_values(ascending=False)

    return mi_scores


# ======================================
# Execute MI calculation
# ======================================

columns_to_exclude = [
    'work_hours_per_day'
]

X_tr_filtered = X_tr.drop(columns=columns_to_exclude, errors='ignore')

mi_scores = get_mi_scores(X_tr_filtered, y_tr)

print("\n--- Mutual Information Scores ---")
print(mi_scores)


# ======================================
# Plot
# ======================================

plt.figure(figsize=(10, 8))

sns.barplot(
    x=mi_scores.values,
    y=mi_scores.index,
    palette="magma"
)

plt.title("Feature Importance (Mutual Information)", fontsize=14)
plt.xlabel("Mutual Information Score")
plt.ylabel("Features")

plt.tight_layout()
plt.show()

In [ ]:
sns.regplot(
    data = df_tr,
    x="work_hours_per_day",
    y="actual_productivity_score",
    lowess=True
)

##uses focus apps

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd


display(df_tr['uses_focus_apps'].value_counts(dropna=False))

# Get unique categories for iteration
categories_focus_apps = df_tr['uses_focus_apps'].unique()
categories_focus_apps = sorted(categories_focus_apps, key=lambda x: str(x))

n_categories = len(categories_focus_apps)
n_cols = 2  # You can adjust the number of columns as needed
n_rows = (n_categories + n_cols - 1) // n_cols # Calculate rows needed

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten()

for i, category in enumerate(categories_focus_apps):
    subset = df_tr[df_tr['uses_focus_apps'] == category]
    sns.histplot(subset['actual_productivity_score'], kde=True, bins=20, ax=axes[i], color='lightgreen')
    axes[i].set_title(f'Productivity Distribution for uses_focus_apps: {category}')
    axes[i].set_xlabel('Actual Productivity Score')
    axes[i].set_ylabel('Frequency')

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()



In [ ]:
from scipy import stats

# --- Independent Samples t-test ---
print("\n--- Independent Samples t-test for uses_focus_apps ---")

group_true = df_tr[df_tr['uses_focus_apps'] == True]['actual_productivity_score'].dropna()
group_false = df_tr[df_tr['uses_focus_apps'] == False]['actual_productivity_score'].dropna()

# Perform t-test
# Equal_var=False for Welch's t-test, which does not assume equal variances
t_statistic, p_value = stats.ttest_ind(group_true, group_false, equal_var=False)

print(f"T-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Conclusion: The p-value is less than 0.05, indicating a statistically significant difference in actual productivity scores between users and non-users of focus apps.")
else:
    print("Conclusion: The p-value is not less than 0.05, indicating no statistically significant difference in actual productivity scores between users and non-users of focus apps.")

In [ ]:
from scipy import stats
import pandas as pd

# --- Mann-Whitney U Test for uses_focus_apps ---
print("\n--- Mann-Whitney U Test for uses_focus_apps ---")

group_true = df_tr[df_tr['uses_focus_apps'] == True]['actual_productivity_score'].dropna()
group_false = df_tr[df_tr['uses_focus_apps'] == False]['actual_productivity_score'].dropna()

# Perform Mann-Whitney U test
u_statistic, p_value = stats.mannwhitneyu(group_true, group_false, alternative='two-sided')

print(f"U-statistic: {u_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Conclusion: The p-value is less than 0.05, indicating a statistically significant difference in actual productivity score distributions between users and non-users of focus apps.")
else:
    print("Conclusion: The p-value is not less than 0.05, indicating no statistically significant difference in actual productivity score distributions between users and non-users of focus apps.")

In [ ]:
from scipy import stats
import pandas as pd

# --- Kolmogorov-Smirnov (KS) Test for uses_focus_apps ---
print("\n--- Kolmogorov-Smirnov (KS) Test for uses_focus_apps ---")

group_true = df_tr[df_tr['uses_focus_apps'] == True]['actual_productivity_score'].dropna()
group_false = df_tr[df_tr['uses_focus_apps'] == False]['actual_productivity_score'].dropna()

# Perform KS 2-sample test
ks_statistic, p_value = stats.ks_2samp(group_true, group_false)

print(f"KS Statistic: {ks_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Conclusion: The p-value is less than 0.05, indicating a statistically significant difference in actual productivity score distributions between users and non-users of focus apps.")
else:
    print("Conclusion: The p-value is not less than 0.05, indicating no statistically significant difference in actual productivity score distributions between users and non-users of focus apps.")

##daily social media time

In [ ]:
df_tr.info()

In [ ]:
print("Rows with daily social media time > 15 hours:")
display(df_tr[df_tr['daily_social_media_time'] > 15])

In [ ]:
# Filter df_tr for individuals with daily social media time > 15 hours
high_social_media_users = df_tr[df_tr['daily_social_media_time'] > 15]

# Get the counts for 'work_hours_per_day_binned' within this filtered group
work_hours_distribution_for_high_social_media_users = high_social_media_users['work_hours_per_day_binned'].value_counts(dropna=False)

print("Count of individuals with >15 hours daily social media time, by work hours binned:")
display(work_hours_distribution_for_high_social_media_users)

In [ ]:
df_tr["extreme_social_media_user"] = (df_tr["daily_social_media_time"] > 12).astype(int)

In [ ]:
from scipy import stats
import pandas as pd

# Ensure 'extreme_social_media_user' is present in df_tr
# Re-create it if df_tr was copied before this feature was added to df_train
if 'extreme_social_media_user' not in df_tr.columns:
    df_tr['extreme_social_media_user'] = (df_tr['daily_social_media_time'] > 12).astype(int)

# --- Mann-Whitney U Test for extreme_social_media_user ---
print("\n--- Mann-Whitney U Test for extreme_social_media_user ---")

group_extreme_user = df_tr[df_tr['extreme_social_media_user'] == 1]['actual_productivity_score'].dropna()
group_non_extreme_user = df_tr[df_tr['extreme_social_media_user'] == 0]['actual_productivity_score'].dropna()

# Perform Mann-Whitney U test
u_statistic, p_value = stats.mannwhitneyu(group_extreme_user, group_non_extreme_user, alternative='two-sided')

print(f"U-statistic: {u_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Conclusion: The p-value is less than 0.05, indicating a statistically significant difference in actual productivity score distributions between extreme social media users and non-extreme users.")
else:
    print("Conclusion: The p-value is not less than 0.05, indicating no statistically significant difference in actual productivity score distributions between extreme social media users and non-extreme users.")

In [ ]:
df_tr["daily_social_media_time"] = df_tr["daily_social_media_time"].clip(upper=12)
# Step 2: Define bins
bins_social_media = [
    0,           # min
    df_tr['daily_social_media_time'].quantile(0.25),  # Q1
    df_tr['daily_social_media_time'].quantile(0.75),  # Q3
    df_tr['daily_social_media_time'].quantile(0.95),  # 95th percentile
    df_tr['daily_social_media_time'].max()            # max
]

labels_social_media = [
    'Very Low',
    'Typical',
    'High',
    'Extreme'
]

# Step 3: Create binned column
df_tr['daily_social_media_time_binned'] = pd.cut(
    df_tr['daily_social_media_time'],
    bins=bins_social_media,
    labels=labels_social_media,
    include_lowest=True
)

# Step 4: Check counts
print("Counts per bin:")
display(df_tr['daily_social_media_time_binned'].value_counts().sort_index())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Get unique categories for iteration
categories_daily_social_media_time_binned = df_tr['daily_social_media_time_binned'].unique()
categories_daily_social_media_time_binned = sorted(categories_daily_social_media_time_binned, key=lambda x: str(x)) # Ensure consistent order

n_categories_sm = len(categories_daily_social_media_time_binned)
n_cols_sm = 2  # Adjust as needed
n_rows_sm = (n_categories_sm + n_cols_sm - 1) // n_cols_sm # Calculate rows needed

fig_sm, axes_sm = plt.subplots(n_rows_sm, n_cols_sm, figsize=(15, 5 * n_rows_sm))
axes_sm = axes_sm.flatten()

for i, category in enumerate(categories_daily_social_media_time_binned):
    subset_sm = df_tr[df_tr['daily_social_media_time_binned'] == category]
    sns.histplot(subset_sm['actual_productivity_score'], kde=True, bins=20, ax=axes_sm[i], color='lightsteelblue')
    axes_sm[i].set_title(f'Productivity Distribution for Social Media Bin: {category}')
    axes_sm[i].set_xlabel('Actual Productivity Score')
    axes_sm[i].set_ylabel('Frequency')

# Hide any unused subplots
for j in range(i + 1, len(axes_sm)):
    fig_sm.delaxes(axes_sm[j])

plt.tight_layout()
plt.show()

In [ ]:
from scipy import stats
import pandas as pd
from itertools import combinations

# Ensure df_tr reflects the latest binning for 'daily_social_media_time_binned'
# This assumes df_train has been updated with the latest binning in cell S2w0964xsChE
# and df_tr is a copy of the updated df_train.
# If df_tr is not up-to-date, re-run feature engineering steps for df_tr.
# For this step, we will assume df_tr is current.

# Get unique categories for iteration
categories_daily_social_media_time_binned = df_tr['daily_social_media_time_binned'].unique()
categories_daily_social_media_time_binned = sorted([cat for cat in categories_daily_social_media_time_binned if pd.notna(cat)], key=lambda x: str(x)) # Ensure consistent order and handle potential NaNs

# --- Kolmogorov-Smirnov (KS) Test for daily_social_media_time_binned ---
print("\n--- Kolmogorov-Smirnov (KS) Test Results for Productivity Distributions (daily_social_media_time_binned) ---")

# Iterate through all unique pairs of categories
for cat1, cat2 in combinations(categories_daily_social_media_time_binned, 2):
    group1 = df_tr[df_tr['daily_social_media_time_binned'] == cat1]['actual_productivity_score'].dropna()
    group2 = df_tr[df_tr['daily_social_media_time_binned'] == cat2]['actual_productivity_score'].dropna()

    # Check if groups have enough data for KS test (at least 2 observations)
    if len(group1) > 1 and len(group2) > 1:
        ks_statistic, p_value = stats.ks_2samp(group1, group2)

        print(f"\nComparing '{cat1}' vs '{cat2}':")
        print(f"  KS Statistic = {ks_statistic:.4f}")
        print(f"  P-value      = {p_value:.4f}")

        if p_value < 0.05:
            print("  Conclusion   = Statistically significant difference in productivity distributions (p < 0.05)")
        else:
            print("  Conclusion   = No statistically significant difference in productivity distributions (p >= 0.05)")
    else:
        print(f"\nSkipping comparison '{cat1}' vs '{cat2}' due to insufficient data.")

In [ ]:
X_tr = df_tr.drop(columns=["actual_productivity_score"])
y_tr = df_tr["actual_productivity_score"]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import LabelEncoder


def get_mi_scores(X, y):

    X_copy = X.copy()

    # -----------------------------
    # 1. Handle categorical columns
    # -----------------------------
    label_encoders = {}

    for col in X_copy.select_dtypes(include=['object', 'category']).columns:
        le = LabelEncoder()
        X_copy[col] = le.fit_transform(X_copy[col].astype(str))
        label_encoders[col] = le

    # -----------------------------
    # 2. Convert boolean → int
    # -----------------------------
    for col in X_copy.select_dtypes(include=['bool']).columns:
        X_copy[col] = X_copy[col].astype(int)

    # -----------------------------
    # 3. Handle missing values
    # -----------------------------
    X_copy = X_copy.fillna(X_copy.median(numeric_only=True))

    # -----------------------------
    # 4. Identify discrete features
    # -----------------------------
    discrete_features = []

    for col in X_copy.columns:
        if (
            pd.api.types.is_integer_dtype(X_copy[col]) or
            X_copy[col].nunique() < 15
        ):
            discrete_features.append(True)
        else:
            discrete_features.append(False)

    # -----------------------------
    # 5. Compute Mutual Information
    # -----------------------------
    print("Calculating MI scores...")

    mi_scores_array = mutual_info_regression(
        X_copy,
        y,
        discrete_features=discrete_features,
        random_state=42,
        n_neighbors=5   # more stable estimation
    )

    mi_scores = pd.Series(
        mi_scores_array,
        index=X_copy.columns,
        name="MI Score"
    ).sort_values(ascending=False)

    return mi_scores


# ======================================
# Execute MI calculation
# ======================================

columns_to_exclude = [
    'work_hours_per_day_binned',
    'daily_social_media_time'
]

X_tr_filtered = X_tr.drop(columns=columns_to_exclude, errors='ignore')

mi_scores = get_mi_scores(X_tr_filtered, y_tr)

print("\n--- Mutual Information Scores ---")
print(mi_scores)


# ======================================
# Plot
# ======================================

plt.figure(figsize=(10, 8))

sns.barplot(
    x=mi_scores.values,
    y=mi_scores.index,
    palette="magma"
)

plt.title("Feature Importance (Mutual Information)", fontsize=14)
plt.xlabel("Mutual Information Score")
plt.ylabel("Features")

plt.tight_layout()
plt.show()

In [ ]:
df_tr = df_tr.drop(columns=["daily_social_media_time_binned"])

In [ ]:
df_tr["social_x_work"] = df_tr["daily_social_media_time"] * df_tr["work_hours_per_day"]

In [ ]:
X_tr = df_tr.drop(columns=["actual_productivity_score"])
y_tr = df_tr["actual_productivity_score"]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import LabelEncoder


def get_mi_scores(X, y):

    X_copy = X.copy()

    # -----------------------------
    # 1. Handle categorical columns
    # -----------------------------
    label_encoders = {}

    for col in X_copy.select_dtypes(include=['object', 'category']).columns:
        le = LabelEncoder()
        X_copy[col] = le.fit_transform(X_copy[col].astype(str))
        label_encoders[col] = le

    # -----------------------------
    # 2. Convert boolean → int
    # -----------------------------
    for col in X_copy.select_dtypes(include=['bool']).columns:
        X_copy[col] = X_copy[col].astype(int)

    # -----------------------------
    # 3. Handle missing values
    # -----------------------------
    X_copy = X_copy.fillna(X_copy.median(numeric_only=True))

    # -----------------------------
    # 4. Identify discrete features
    # -----------------------------
    discrete_features = []

    for col in X_copy.columns:
        if (
            pd.api.types.is_integer_dtype(X_copy[col]) or
            X_copy[col].nunique() < 15
        ):
            discrete_features.append(True)
        else:
            discrete_features.append(False)

    # -----------------------------
    # 5. Compute Mutual Information
    # -----------------------------
    print("Calculating MI scores...")

    mi_scores_array = mutual_info_regression(
        X_copy,
        y,
        discrete_features=discrete_features,
        random_state=42,
        n_neighbors=5   # more stable estimation
    )

    mi_scores = pd.Series(
        mi_scores_array,
        index=X_copy.columns,
        name="MI Score"
    ).sort_values(ascending=False)

    return mi_scores


# ======================================
# Execute MI calculation
# ======================================

columns_to_exclude = [
    'work_hours_per_day_binned',
    'daily_social_media_time_binned'
]

X_tr_filtered = X_tr.drop(columns=columns_to_exclude, errors='ignore')

mi_scores = get_mi_scores(X_tr_filtered, y_tr)

print("\n--- Mutual Information Scores ---")
print(mi_scores)


# ======================================
# Plot
# ======================================

plt.figure(figsize=(10, 8))

sns.barplot(
    x=mi_scores.values,
    y=mi_scores.index,
    palette="magma"
)

plt.title("Feature Importance (Mutual Information)", fontsize=14)
plt.xlabel("Mutual Information Score")
plt.ylabel("Features")

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
sns.scatterplot(
    x=df_tr['social_x_work'],
    y=df_tr['actual_productivity_score']
)

In [ ]:
df_tr =  df_tr.drop(columns=["social_x_work"])

##job type

In [ ]:
job_dummies = pd.get_dummies(
    df_tr["job_type"],
    prefix="job",
    drop_first=True
)
for col in job_dummies.columns:
    df_tr[f"{col}_x_work"] = (
        job_dummies[col] * df_tr["work_hours_per_day"]
    )
df_tr = pd.concat([df_tr, job_dummies], axis=1)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get unique categories for iteration
categories_job_type = df_tr['job_type'].unique()
categories_job_type = sorted(categories_job_type, key=lambda x: str(x)) # Ensure consistent order

n_categories_job_type = len(categories_job_type)
n_cols_job_type = 2  # Adjust as needed
n_rows_job_type = (n_categories_job_type + n_cols_job_type - 1) // n_cols_job_type # Calculate rows needed

fig_job_type, axes_job_type = plt.subplots(n_rows_job_type, n_cols_job_type, figsize=(15, 5 * n_rows_job_type))
axes_job_type = axes_job_type.flatten()

for i, category in enumerate(categories_job_type):
    subset_job_type = df_tr[df_tr['job_type'] == category]
    sns.histplot(subset_job_type['actual_productivity_score'], kde=True, bins=20, ax=axes_job_type[i], color='lightsteelblue')
    axes_job_type[i].set_title(f'Productivity Distribution for Job Type: {category}')
    axes_job_type[i].set_xlabel('Actual Productivity Score')
    axes_job_type[i].set_ylabel('Frequency')

# Hide any unused subplots
for j in range(i + 1, len(axes_job_type)):
    fig_job_type.delaxes(axes_job_type[j])

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations

# Get unique categories for iteration (from the existing df_tr)
categories_job_type = df_tr['job_type'].unique()
categories_job_type = sorted(categories_job_type, key=lambda x: str(x))

# --- ANOVA Test ---
print("\n--- ANOVA Test Results (job_type) ---")

# Prepare data for ANOVA
groups_anova_job_type = []
for category in categories_job_type:
    group_data = df_tr[df_tr['job_type'] == category]['actual_productivity_score'].dropna()
    if not group_data.empty: # Only add non-empty groups
        groups_anova_job_type.append(group_data)

if len(groups_anova_job_type) > 1:
    f_statistic, p_value_anova = stats.f_oneway(*groups_anova_job_type)

    print(f"F-statistic: {f_statistic:.4f}")
    print(f"P-value: {p_value_anova:.4f}")

    if p_value_anova < 0.05:
        print("Conclusion: The p-value is less than 0.05, indicating a statistically significant difference in actual productivity scores among these job type categories.")
    else:
        print("Conclusion: The p-value is not less than 0.05, indicating no statistically significant difference in actual productivity scores among these job type categories.")
else:
    print("Not enough groups with data to perform ANOVA test.")

# --- Kolmogorov-Smirnov (KS) Test for pairs of categories ---
print("\n--- Kolmogorov-Smirnov (KS) Test Results for Productivity Distributions (job_type) ---")

# Iterate through all unique pairs of categories
for cat1, cat2 in combinations(categories_job_type, 2):
    group1_ks = df_tr[df_tr['job_type'] == cat1]['actual_productivity_score'].dropna()
    group2_ks = df_tr[df_tr['job_type'] == cat2]['actual_productivity_score'].dropna()

    # Check if groups have enough data for KS test (at least 2 observations)
    if len(group1_ks) > 1 and len(group2_ks) > 1:
        ks_statistic, p_value_ks = stats.ks_2samp(group1_ks, group2_ks)

        print(f"\nComparing '{cat1}' vs '{cat2}':")
        print(f"  KS Statistic = {ks_statistic:.4f}")
        print(f"  P-value      = {p_value_ks:.4f}")

        if p_value_ks < 0.05:
            print("  Conclusion   = Statistically significant difference in productivity distributions (p < 0.05)")
        else:
            print("  Conclusion   = No statistically significant difference in productivity distributions (p >= 0.05)")
    else:
        print(f"\nSkipping comparison '{cat1}' vs '{cat2}' due to insufficient data.")

In [ ]:
# --- Kruskal–Wallis Test ---
print("\n--- Kruskal–Wallis Test Results (job_type) ---")

# Prepare data for Kruskal–Wallis
groups_kw_job_type = []

for category in categories_job_type:

    group_data = df_tr[
        df_tr['job_type'] == category
    ]['actual_productivity_score'].dropna()

    if not group_data.empty:   # only include non-empty groups
        groups_kw_job_type.append(group_data)


# Perform test only if enough groups exist
if len(groups_kw_job_type) > 1:

    h_statistic, p_value_kw = stats.kruskal(*groups_kw_job_type)

    print(f"H-statistic: {h_statistic:.4f}")
    print(f"P-value: {p_value_kw:.4f}")

    if p_value_kw < 0.05:

        print("Conclusion: The p-value is less than 0.05, indicating a statistically significant difference in productivity among job type categories.")

    else:

        print("Conclusion: The p-value is not less than 0.05, indicating no statistically significant difference in productivity among job type categories.")

else:

    print("Not enough groups with data to perform Kruskal–Wallis test.")

In [ ]:
X_tr  =  df_tr.drop(columns=["actual_productivity_score"])
y_tr = df_tr["actual_productivity_score"]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import LabelEncoder


def get_mi_scores(X, y):

    X_copy = X.copy()

    # -----------------------------
    # 1. Handle categorical columns
    # -----------------------------
    label_encoders = {}

    for col in X_copy.select_dtypes(include=['object', 'category']).columns:
        le = LabelEncoder()
        X_copy[col] = le.fit_transform(X_copy[col].astype(str))
        label_encoders[col] = le

    # -----------------------------
    # 2. Convert boolean → int
    # -----------------------------
    for col in X_copy.select_dtypes(include=['bool']).columns:
        X_copy[col] = X_copy[col].astype(int)

    # -----------------------------
    # 3. Handle missing values
    # -----------------------------
    X_copy = X_copy.fillna(X_copy.median(numeric_only=True))

    # -----------------------------
    # 4. Identify discrete features
    # -----------------------------
    discrete_features = []

    for col in X_copy.columns:
        if (
            pd.api.types.is_integer_dtype(X_copy[col]) or
            X_copy[col].nunique() < 15
        ):
            discrete_features.append(True)
        else:
            discrete_features.append(False)

    # -----------------------------
    # 5. Compute Mutual Information
    # -----------------------------
    print("Calculating MI scores...")

    mi_scores_array = mutual_info_regression(
        X_copy,
        y,
        discrete_features=discrete_features,
        random_state=42,
        n_neighbors=5   # more stable estimation
    )

    mi_scores = pd.Series(
        mi_scores_array,
        index=X_copy.columns,
        name="MI Score"
    ).sort_values(ascending=False)

    return mi_scores


# ======================================
# Execute MI calculation
# ======================================

columns_to_exclude = [
    'extreme_social_media_user',
    'screen_time_before_sleep_binned',
    'work_hours_per_day_binned'
]

X_tr_filtered = X_tr.drop(columns=columns_to_exclude, errors='ignore')

mi_scores = get_mi_scores(X_tr_filtered, y_tr)

print("\n--- Mutual Information Scores ---")
print(mi_scores)


# ======================================
# Plot
# ======================================

plt.figure(figsize=(10, 8))

sns.barplot(
    x=mi_scores.values,
    y=mi_scores.index,
    palette="magma"
)

plt.title("Feature Importance (Mutual Information)", fontsize=14)
plt.xlabel("Mutual Information Score")
plt.ylabel("Features")

plt.tight_layout()
plt.show()

In [ ]:
# ====== JOB DUMMIES + INTERACTIONS FOR TEST (df_ts) ======

# 1) Create test dummies (same rule: drop_first=True)
job_dummies_ts = pd.get_dummies(
    df_ts["job_type"],
    prefix="job",
    drop_first=True
)

# 2) Force df_ts to have the SAME dummy columns as df_tr had
# (important: use df_tr's dummy columns as the reference)
job_dummy_cols_tr = job_dummies.columns  # from your train code
job_dummies_ts = job_dummies_ts.reindex(columns=job_dummy_cols_tr, fill_value=0)

# 3) Create interaction terms on df_ts using those aligned dummy columns
for col in job_dummy_cols_tr:
    df_ts[f"{col}_x_work"] = job_dummies_ts[col] * df_ts["work_hours_per_day"]

# 4) Concatenate the aligned dummies into df_ts
df_ts = pd.concat([df_ts, job_dummies_ts], axis=1)

In [ ]:
missing_in_ts = set(df_tr.columns) - set(df_ts.columns)
extra_in_ts   = set(df_ts.columns) - set(df_tr.columns)

print("Missing in df_ts:", missing_in_ts)
print("Extra in df_ts:", extra_in_ts)

##age

In [ ]:
df_tr['age_squared'] = df_tr['age'] ** 2

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------
# Model 1: Linear
# ----------------------------

X_linear = df_tr[['age']]
X_linear = sm.add_constant(X_linear)

y = df_tr['actual_productivity_score']

model_linear = sm.OLS(y, X_linear).fit()

print("\nLINEAR MODEL SUMMARY")
print(model_linear.summary())


# ----------------------------
# Model 2: Polynomial
# ----------------------------

X_poly = df_tr[['age', 'age_squared']]
X_poly = sm.add_constant(X_poly)

model_poly = sm.OLS(y, X_poly).fit()

print("\nPOLYNOMIAL MODEL SUMMARY")
print(model_poly.summary())

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.api import het_breuschpagan


# ----------------------------------------
# Use residuals from the NEW polynomial model
# ----------------------------------------

residuals = model_poly.resid

# exogenous variables used in the model (including constant)
X_bp = model_poly.model.exog


# ----------------------------------------
# Anderson–Darling Test
# ----------------------------------------

print("\n--- Anderson-Darling Test for Normality of Residuals ---")

result_ad = stats.anderson(residuals, dist='norm')

print(f"Statistic: {result_ad.statistic:.4f}")

print("Critical Values:")

for sl, cv in zip(result_ad.significance_level, result_ad.critical_values):

    print(f"  {sl:3.1f}%: {cv:.4f} (Reject H0 if Statistic >= Critical Value)")


# Check against 5% level
cv_5 = result_ad.critical_values[
    list(result_ad.significance_level).index(5.0)
]

if result_ad.statistic < cv_5:

    print("Conclusion: Residuals appear normally distributed (Fail to reject H0 at 5%).")

else:

    print("Conclusion: Residuals are NOT normally distributed (Reject H0 at 5%).")



# ----------------------------------------
# Breusch–Pagan Test
# ----------------------------------------

print("\n--- Breusch-Pagan Test for Heteroscedasticity ---")

lm_stat, lm_p_value, f_stat, f_p_value = het_breuschpagan(

    residuals,
    X_bp
)

print(f"Lagrange Multiplier Statistic: {lm_stat:.4f}")
print(f"LM p-value: {lm_p_value:.4f}")
print(f"F Statistic: {f_stat:.4f}")
print(f"F p-value: {f_p_value:.4f}")


if lm_p_value < 0.05:

    print("Conclusion: Heteroscedasticity detected (Reject H0).")

else:

    print("Conclusion: Homoscedasticity assumed (Fail to reject H0).")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import numpy as np

# Residuals and fitted values from statsmodels polynomial model
residuals = model_poly.resid
fitted = model_poly.fittedvalues

# Standardized residuals
standardized_residuals = residuals / np.std(residuals)

# Create 2x2 canvas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))


# -----------------------------------
# 1. Residuals vs Fitted
# -----------------------------------

sns.scatterplot(
    x=fitted,
    y=residuals,
    ax=axes[0,0],
    alpha=0.6
)

axes[0,0].axhline(0, color='red', linestyle='--')

axes[0,0].set_title("Residuals vs Fitted")
axes[0,0].set_xlabel("Fitted Values")
axes[0,0].set_ylabel("Residuals")


# -----------------------------------
# 2. Histogram of Residuals
# -----------------------------------

sns.histplot(
    residuals,
    kde=True,
    ax=axes[0,1]
)

axes[0,1].set_title("Histogram of Residuals")
axes[0,1].set_xlabel("Residuals")


# -----------------------------------
# 3. QQ Plot
# -----------------------------------

stats.probplot(
    residuals,
    dist="norm",
    plot=axes[1,0]
)

axes[1,0].set_title("QQ Plot")


# -----------------------------------
# 4. Scale-Location Plot
# -----------------------------------

sns.scatterplot(
    x=fitted,
    y=np.sqrt(np.abs(standardized_residuals)),
    ax=axes[1,1],
    alpha=0.6
)

axes[1,1].set_title("Scale-Location Plot")
axes[1,1].set_xlabel("Fitted Values")
axes[1,1].set_ylabel("√|Standardized Residuals|")


plt.tight_layout()

plt.show()

In [ ]:
df_tr.info()

In [ ]:
X_tr = df_tr.drop(columns=["actual_productivity_score"])
y_tr = df_tr["actual_productivity_score"]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import LabelEncoder


def get_mi_scores(X, y):

    X_copy = X.copy()

    # -----------------------------
    # 1. Handle categorical columns
    # -----------------------------
    label_encoders = {}

    for col in X_copy.select_dtypes(include=['object', 'category']).columns:
        le = LabelEncoder()
        X_copy[col] = le.fit_transform(X_copy[col].astype(str))
        label_encoders[col] = le

    # -----------------------------
    # 2. Convert boolean → int
    # -----------------------------
    for col in X_copy.select_dtypes(include=['bool']).columns:
        X_copy[col] = X_copy[col].astype(int)

    # -----------------------------
    # 3. Handle missing values
    # -----------------------------
    X_copy = X_copy.fillna(X_copy.median(numeric_only=True))

    # -----------------------------
    # 4. Identify discrete features
    # -----------------------------
    discrete_features = []

    for col in X_copy.columns:
        if (
            pd.api.types.is_integer_dtype(X_copy[col]) or
            X_copy[col].nunique() < 15
        ):
            discrete_features.append(True)
        else:
            discrete_features.append(False)

    # -----------------------------
    # 5. Compute Mutual Information
    # -----------------------------
    print("Calculating MI scores...")

    mi_scores_array = mutual_info_regression(
        X_copy,
        y,
        discrete_features=discrete_features,
        random_state=42,
        n_neighbors=5   # more stable estimation
    )

    mi_scores = pd.Series(
        mi_scores_array,
        index=X_copy.columns,
        name="MI Score"
    ).sort_values(ascending=False)

    return mi_scores


# ======================================
# Execute MI calculation
# ======================================

columns_to_exclude = [
    'extreme_social_media_user',
    'screen_time_before_sleep_binned',
    'work_hours_per_day_binned'
]

X_tr_filtered = X_tr.drop(columns=columns_to_exclude, errors='ignore')

mi_scores = get_mi_scores(X_tr_filtered, y_tr)

print("\n--- Mutual Information Scores ---")
print(mi_scores)


# ======================================
# Plot
# ======================================

plt.figure(figsize=(10, 8))

sns.barplot(
    x=mi_scores.values,
    y=mi_scores.index,
    palette="magma"
)

plt.title("Feature Importance (Mutual Information)", fontsize=14)
plt.xlabel("Mutual Information Score")
plt.ylabel("Features")

plt.tight_layout()
plt.show()

In [ ]:
df_tr = df_tr.drop(columns=["age_squared"])

##days feeling burnout per month

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.regplot(
    x=df_tr['days_feeling_burnout_per_month'],
    y=df_tr['actual_productivity_score'],
    scatter_kws={'alpha':0.3},
    line_kws={'color':'red'}
)

plt.title("Burnout Days vs Productivity")
plt.show()

In [ ]:
df_tr['burnout_sq'] = (
    df_tr['days_feeling_burnout_per_month'] ** 2
)

In [ ]:
get_mi_scores(
    df_tr[['days_feeling_burnout_per_month','burnout_sq']],
    y_tr
)

In [ ]:
bins = [0, 5, 15, 25, 31]
labels = ['Low', 'Moderate', 'High', 'Severe']

df_tr['burnout_binned'] = pd.cut(
    df_tr['days_feeling_burnout_per_month'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

df_tr['burnout_binned'].value_counts().sort_index()

In [ ]:
sns.boxplot(
    x='burnout_binned',
    y='actual_productivity_score',
    data=df_tr
)
plt.show()

In [ ]:
from scipy.stats import kruskal

groups = [
    g['actual_productivity_score'].values
    for _, g in df_tr.groupby('burnout_binned', observed=False)
]

kruskal_result = kruskal(*groups)

print(f"\n--- Kruskal-Wallis Test Results ---")
print(f"H-statistic: {kruskal_result.statistic:.4f}")
print(f"P-value:     {kruskal_result.pvalue:.4f}")

if kruskal_result.pvalue < 0.05:
    print("Conclusion: The p-value is less than 0.05, indicating a statistically significant difference in actual productivity scores among the burnout bins.")
else:
    print("Conclusion: The p-value is not less than 0.05, indicating no statistically significant difference in actual productivity scores among the burnout bins.")

In [ ]:
df_tr = df_tr.drop(columns=["burnout_binned","burnout_sq"])

##weekly offline hours

In [ ]:
sns.regplot(
    x=df_tr['weekly_offline_hours'],
    y=df_tr['actual_productivity_score'],
    scatter_kws={'alpha':0.3},
    line_kws={'color':'red'}
)
plt.title("Weekly Offline Hours vs Productivity")
plt.show()

In [ ]:
# Squared term
df_tr['weekly_offline_hours_sq'] = df_tr['weekly_offline_hours'] ** 2

# Log transform (only if slight skew is a problem)
import numpy as np
df_tr['weekly_offline_hours_log'] = np.log1p(df_tr['weekly_offline_hours'])

In [ ]:
bins = [
    df_tr['weekly_offline_hours'].min(),
    df_tr['weekly_offline_hours'].quantile(0.25),
    df_tr['weekly_offline_hours'].quantile(0.75),
    df_tr['weekly_offline_hours'].max()
]

labels = ['Low', 'Medium', 'High']

df_tr['weekly_offline_hours_binned'] = pd.cut(
    df_tr['weekly_offline_hours'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

df_tr['weekly_offline_hours_binned'].value_counts()

In [ ]:
sns.boxplot(
    x='weekly_offline_hours_binned',
    y='actual_productivity_score',
    data=df_tr
)
plt.show()

In [ ]:
X_tr = df_tr.drop(columns=["actual_productivity_score"])
y_tr = df_tr["actual_productivity_score"]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import LabelEncoder


def get_mi_scores(X, y):

    X_copy = X.copy()

    # -----------------------------
    # 1. Handle categorical columns
    # -----------------------------
    label_encoders = {}

    for col in X_copy.select_dtypes(include=['object', 'category']).columns:
        le = LabelEncoder()
        X_copy[col] = le.fit_transform(X_copy[col].astype(str))
        label_encoders[col] = le

    # -----------------------------
    # 2. Convert boolean → int
    # -----------------------------
    for col in X_copy.select_dtypes(include=['bool']).columns:
        X_copy[col] = X_copy[col].astype(int)

    # -----------------------------
    # 3. Handle missing values
    # -----------------------------
    X_copy = X_copy.fillna(X_copy.median(numeric_only=True))

    # -----------------------------
    # 4. Identify discrete features
    # -----------------------------
    discrete_features = []

    for col in X_copy.columns:
        if (
            pd.api.types.is_integer_dtype(X_copy[col]) or
            X_copy[col].nunique() < 15
        ):
            discrete_features.append(True)
        else:
            discrete_features.append(False)

    # -----------------------------
    # 5. Compute Mutual Information
    # -----------------------------
    print("Calculating MI scores...")

    mi_scores_array = mutual_info_regression(
        X_copy,
        y,
        discrete_features=discrete_features,
        random_state=42,
        n_neighbors=5   # more stable estimation
    )

    mi_scores = pd.Series(
        mi_scores_array,
        index=X_copy.columns,
        name="MI Score"
    ).sort_values(ascending=False)

    return mi_scores


# ======================================
# Execute MI calculation
# ======================================

columns_to_exclude = [
    'extreme_social_media_user',
    'screen_time_before_sleep_binned',
    'work_hours_per_day_binned',
    "weekly_offline_hours",
    "weekly_offline_hours_sq"
]

X_tr_filtered = X_tr.drop(columns=columns_to_exclude, errors='ignore')

mi_scores = get_mi_scores(X_tr_filtered, y_tr)

print("\n--- Mutual Information Scores ---")
print(mi_scores)


# ======================================
# Plot
# ======================================

plt.figure(figsize=(10, 8))

sns.barplot(
    x=mi_scores.values,
    y=mi_scores.index,
    palette="magma"
)

plt.title("Feature Importance (Mutual Information)", fontsize=14)
plt.xlabel("Mutual Information Score")
plt.ylabel("Features")

plt.tight_layout()
plt.show()

In [ ]:
df_tr =  df_tr.drop(columns=["weekly_offline_hours_binned","weekly_offline_hours_sq","weekly_offline_hours_log"])

In [ ]:
df_tr.info()

##coffee consumption per day

In [ ]:
import pandas as pd

bins = [-1, 0, 2, 4, df_tr['coffee_consumption_per_day'].max()]
labels = ['None', 'Low', 'Medium', 'High']

df_tr['coffee_binned'] = pd.cut(
    df_tr['coffee_consumption_per_day'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

df_tr['coffee_binned'].value_counts().sort_index()

In [ ]:
import seaborn as sns
sns.boxplot(
    x='coffee_binned',
    y='actual_productivity_score',
    data=df_tr
)
plt.show()

In [ ]:
df_tr = df_tr.drop(columns=["coffee_binned"])

##job satisfaction score

In [ ]:
df_tr.info()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.regplot(
    x=df_tr['job_satisfaction_score'],
    y=df_tr['actual_productivity_score'],
    scatter_kws={'alpha':0.3},
    line_kws={'color':'red'}
)
plt.title("Job Satisfaction vs Productivity")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Create polynomial term
df_tr['job_satisfaction_squared'] = df_tr['job_satisfaction_score'] ** 2

y = df_tr['actual_productivity_score']

# -------------------------
# Linear model
# -------------------------

X_linear = df_tr[['job_satisfaction_score']]
X_linear = sm.add_constant(X_linear)

model_linear_js = sm.OLS(y, X_linear).fit()

print("\nLINEAR MODEL SUMMARY (Job Satisfaction)")
print(model_linear_js.summary())


# -------------------------
# Polynomial model
# -------------------------

X_poly = df_tr[['job_satisfaction_score', 'job_satisfaction_squared']]
X_poly = sm.add_constant(X_poly)

model_poly_js = sm.OLS(y, X_poly).fit()

print("\nPOLYNOMIAL MODEL SUMMARY (Job Satisfaction)")
print(model_poly_js.summary())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

residuals = model_poly_js.resid
fitted = model_poly_js.fittedvalues

standardized_residuals = residuals / np.std(residuals)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))


# Residual vs fitted
sns.scatterplot(x=fitted, y=residuals, ax=axes[0,0])
axes[0,0].axhline(0, color='red', linestyle='--')
axes[0,0].set_title("Residuals vs Fitted")


# Histogram
sns.histplot(residuals, kde=True, ax=axes[0,1])
axes[0,1].set_title("Histogram")


# QQ plot
stats.probplot(residuals, dist="norm", plot=axes[1,0])
axes[1,0].set_title("QQ Plot")


# Scale-location
sns.scatterplot(
    x=fitted,
    y=np.sqrt(np.abs(standardized_residuals)),
    ax=axes[1,1]
)
axes[1,1].set_title("Scale-Location")


plt.tight_layout()
plt.show()

In [ ]:
import scipy.stats as stats

print("\n--- Anderson-Darling Test ---")

result_ad = stats.anderson(residuals, dist='norm')

print("Statistic:", result_ad.statistic)

for sl, cv in zip(result_ad.significance_level, result_ad.critical_values):

    print(f"{sl}%: {cv}")

if result_ad.statistic < result_ad.critical_values[2]:

    print("Residuals are normal")

else:

    print("Residuals are NOT normal")

In [ ]:
from statsmodels.stats.api import het_breuschpagan

print("\n--- Breusch-Pagan Test ---")

lm, lm_pvalue, f, f_pvalue = het_breuschpagan(
    residuals,
    model_poly_js.model.exog
)

print("LM p-value:", lm_pvalue)

if lm_pvalue < 0.05:

    print("Heteroscedasticity present")

else:

    print("Constant variance")

In [ ]:
from statsmodels.stats.anova import anova_lm

anova_results = anova_lm(model_linear_js, model_poly_js)

print(anova_results)

In [ ]:
df_tr = df_tr.drop(columns=["job_satisfaction_squared"])

#Modeling

##set up dataset

In [ ]:
df_tr = df_tr.drop(columns=['work_hours_per_day_binned', 'extreme_social_media_user'])

In [ ]:
df_tr.info()

In [ ]:
df_ts = df_ts.reindex(columns=df_tr.columns)

In [ ]:
df_ts.info()

In [ ]:
df_tr.to_csv("df_tr.csv", index=False)
df_ts.to_csv("df_ts.csv", index=False)

##Linear Regresion

In [ ]:
target = "actual_productivity_score"

# Build X/y (make sure these are the ones you use everywhere)
X_tr_lr = df_tr.drop(columns=[target]).copy()
y_tr_lr = df_tr[target].copy()

X_ts_lr = df_ts.drop(columns=[target]).copy()
y_ts_lr = df_ts[target].copy()

# ✅ enforce same columns/order
X_ts_lr = X_ts_lr.reindex(columns=X_tr_lr.columns)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Identify categorical columns (category or object)
cat_cols = X_tr_lr.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X_tr_lr.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

model = Pipeline(steps=[
    ("prep", preprocess),
    ("lr", LinearRegression())
])

# Fit
model.fit(X_tr_lr, y_tr_lr)

# Predict
y_pred_tr = model.predict(X_tr_lr)
y_pred_ts = model.predict(X_ts_lr)

# Metrics
print("TRAIN RESULTS")
print("R2:", r2_score(y_tr_lr, y_pred_tr))
print("MAE:", mean_absolute_error(y_tr_lr, y_pred_tr))
print("RMSE:", np.sqrt(mean_squared_error(y_tr_lr, y_pred_tr)))

print("\nTEST RESULTS")
print("R2:", r2_score(y_ts_lr, y_pred_ts))
print("MAE:", mean_absolute_error(y_ts_lr, y_pred_ts))
print("RMSE:", np.sqrt(mean_squared_error(y_ts_lr, y_pred_ts)))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

# =====================================
# Residual Plots (Train) — Pipeline Version
# Uses: model (Pipeline), X_tr_lr, y_tr_lr
# Target column: actual_productivity_score
# =====================================

# Predict on train (IMPORTANT: use the PIPELINE model)
y_pred_tr = model.predict(X_tr_lr)

# Residuals
residuals = y_tr_lr - y_pred_tr

# 1) Residuals vs Predicted
plt.figure(figsize=(9, 5))
plt.scatter(y_pred_tr, residuals, alpha=0.5)
plt.axhline(0)
plt.xlabel("Predicted")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("Residuals vs Predicted (Train)")
plt.tight_layout()
plt.show()

# 2) Residual Histogram
plt.figure(figsize=(9, 5))
plt.hist(residuals, bins=50)
plt.xlabel("Residual")
plt.ylabel("Count")
plt.title("Residual Distribution (Train)")
plt.tight_layout()
plt.show()

# 3) Q-Q Plot
plt.figure(figsize=(9, 5))
stats.probplot(residuals, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals (Train)")
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.stats.diagnostic import normal_ad

stat, p_value = normal_ad(residuals)

print("Anderson-Darling Test")

print("Statistic:", stat)
print("p-value:", p_value)

if p_value > 0.05:
    print("Residuals are NORMAL")
else:
    print("Residuals are NOT normal")

In [ ]:
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan

# 1) Preprocessor from pipeline
prep = model.named_steps["prep"]

# 2) Transform (after OHE)
X_tr_enc = prep.transform(X_tr_lr)

# 3) Convert sparse -> dense if needed
if hasattr(X_tr_enc, "toarray"):
    X_tr_enc = X_tr_enc.toarray()

# ✅ 4) Force numeric float dtype (fixes object/bool issues)
X_tr_enc = np.asarray(X_tr_enc, dtype=np.float64)

# ✅ 5) Check / handle NaN or Inf (statsmodels will crash otherwise)
if not np.isfinite(X_tr_enc).all():
    bad = ~np.isfinite(X_tr_enc)
    print("Non-finite values found:", bad.sum())
    # Replace inf with nan, then nan with 0 (simple, safe for diagnostics)
    X_tr_enc = np.nan_to_num(X_tr_enc, nan=0.0, posinf=0.0, neginf=0.0)

# 6) Add constant
X_tr_enc_const = sm.add_constant(X_tr_enc, has_constant="add")

# 7) Fit OLS
y = np.asarray(y_tr_lr, dtype=np.float64)
ols = sm.OLS(y, X_tr_enc_const).fit()

# 8) Breusch–Pagan
lm_stat, lm_pvalue, f_stat, f_pvalue = het_breuschpagan(ols.resid, ols.model.exog)

print("Breusch-Pagan Test")
print("LM stat:", lm_stat)
print("LM p-value:", lm_pvalue)
print("F stat:", f_stat)
print("F p-value:", f_pvalue)

if lm_pvalue > 0.05:
    print("✅ Homoscedasticity satisfied (fail to reject heteroscedasticity)")
else:
    print("❌ Heteroscedasticity present (reject homoscedasticity)")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Create evaluation table
evaluation_table = pd.DataFrame({

    "Dataset": ["Train", "Test"],

    "MAE": [
        mean_absolute_error(y_tr_lr, y_pred_tr),
        mean_absolute_error(y_ts_lr, y_pred_ts)
    ],

    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_lr, y_pred_tr)),
        np.sqrt(mean_squared_error(y_ts_lr, y_pred_ts))
    ],

    "R²": [
        r2_score(y_tr_lr, y_pred_tr),
        r2_score(y_ts_lr, y_pred_ts)
    ]

})

# Round for clean display
evaluation_table = evaluation_table.round(4)

evaluation_table

##Ridge Regresion

In [ ]:
# =========================================================
# Copies
# =========================================================
target = "actual_productivity_score"

X_tr_ridge = df_tr.drop(columns=[target]).copy()
y_tr_ridge = df_tr[target].copy()

X_ts_ridge = df_ts.drop(columns=[target]).copy()
y_ts_ridge = df_ts[target].copy()

X_ts_ridge = X_ts_ridge.reindex(columns=X_tr_ridge.columns)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# =========================================================
# Preprocessing (Scaling needed!)
# =========================================================
cat_cols = X_tr_ridge.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X_tr_ridge.columns if c not in cat_cols]

preprocess_ridge = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols)
])

# =========================================================
# Model
# =========================================================
ridge = Ridge(alpha=1.0)

ridge_model = Pipeline([
    ("prep", preprocess_ridge),
    ("ridge", ridge)
])

# Fit
ridge_model.fit(X_tr_ridge, y_tr_ridge)

# Predict
y_pred_tr_ridge = ridge_model.predict(X_tr_ridge)
y_pred_ts_ridge = ridge_model.predict(X_ts_ridge)

# Evaluation
ridge_eval = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [
        mean_absolute_error(y_tr_ridge, y_pred_tr_ridge),
        mean_absolute_error(y_ts_ridge, y_pred_ts_ridge)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_ridge, y_pred_tr_ridge)),
        np.sqrt(mean_squared_error(y_ts_ridge, y_pred_ts_ridge))
    ],
    "R²": [
        r2_score(y_tr_ridge, y_pred_tr_ridge),
        r2_score(y_ts_ridge, y_pred_ts_ridge)
    ]
}).round(4).set_index("Dataset")

ridge_eval

##Lasso Regresion

In [ ]:
# =========================================================
# Copies
# =========================================================
X_tr_lasso = df_tr.drop(columns=[target]).copy()
y_tr_lasso = df_tr[target].copy()

X_ts_lasso = df_ts.drop(columns=[target]).copy()
y_ts_lasso = df_ts[target].copy()

X_ts_lasso = X_ts_lasso.reindex(columns=X_tr_lasso.columns)

In [ ]:
from sklearn.linear_model import Lasso

# =========================================================
# Model
# =========================================================
lasso = Lasso(alpha=0.01, max_iter=10000)

lasso_model = Pipeline([
    ("prep", preprocess_ridge),  # same preprocessing
    ("lasso", lasso)
])

# Fit
lasso_model.fit(X_tr_lasso, y_tr_lasso)

# Predict
y_pred_tr_lasso = lasso_model.predict(X_tr_lasso)
y_pred_ts_lasso = lasso_model.predict(X_ts_lasso)

# Evaluation
lasso_eval = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [
        mean_absolute_error(y_tr_lasso, y_pred_tr_lasso),
        mean_absolute_error(y_ts_lasso, y_pred_ts_lasso)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_lasso, y_pred_tr_lasso)),
        np.sqrt(mean_squared_error(y_ts_lasso, y_pred_ts_lasso))
    ],
    "R²": [
        r2_score(y_tr_lasso, y_pred_tr_lasso),
        r2_score(y_ts_lasso, y_pred_ts_lasso)
    ]
}).round(4).set_index("Dataset")

lasso_eval

##ElasticNet

In [ ]:
import numpy as np
import pandas as pd

target = "actual_productivity_score"

X_tr_en = df_tr.drop(columns=[target]).copy()
y_tr_en = df_tr[target].copy()

X_ts_en = df_ts.drop(columns=[target]).copy()
y_ts_en = df_ts[target].copy()

# keep same columns/order (safe)
X_ts_en = X_ts_en.reindex(columns=X_tr_en.columns)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# categorical columns
cat_cols = X_tr_en.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X_tr_en.columns if c not in cat_cols]

preprocess_en = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop"
)

elastic_net = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42)

en_model = Pipeline(steps=[
    ("prep", preprocess_en),
    ("scale", StandardScaler(with_mean=False)),  # ✅ needed for regularization
    ("en", elastic_net)
])

en_model.fit(X_tr_en, y_tr_en)

y_pred_tr_en = en_model.predict(X_tr_en)
y_pred_ts_en = en_model.predict(X_ts_en)

en_eval = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [
        mean_absolute_error(y_tr_en, y_pred_tr_en),
        mean_absolute_error(y_ts_en, y_pred_ts_en)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_en, y_pred_tr_en)),
        np.sqrt(mean_squared_error(y_ts_en, y_pred_ts_en))
    ],
    "R²": [
        r2_score(y_tr_en, y_pred_tr_en),
        r2_score(y_ts_en, y_pred_ts_en)
    ]
}).round(4).set_index("Dataset")

en_eval

In [ ]:
from sklearn.linear_model import ElasticNetCV

en_cv = Pipeline(steps=[
    ("prep", preprocess_en),
    ("scale", StandardScaler(with_mean=False)),
    ("en", ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
        alphas=np.logspace(-4, 1, 25),
        cv=5,
        random_state=42,
        n_jobs=-1
    ))
])

en_cv.fit(X_tr_en, y_tr_en)

best_alpha = en_cv.named_steps["en"].alpha_
best_l1_ratio = en_cv.named_steps["en"].l1_ratio_

print("Best alpha:", best_alpha)
print("Best l1_ratio:", best_l1_ratio)

y_pred_tr_en_tuned = en_cv.predict(X_tr_en)
y_pred_ts_en_tuned = en_cv.predict(X_ts_en)

en_tuned_eval = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [
        mean_absolute_error(y_tr_en, y_pred_tr_en_tuned),
        mean_absolute_error(y_ts_en, y_pred_ts_en_tuned)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_en, y_pred_tr_en_tuned)),
        np.sqrt(mean_squared_error(y_ts_en, y_pred_ts_en_tuned))
    ],
    "R²": [
        r2_score(y_tr_en, y_pred_tr_en_tuned),
        r2_score(y_ts_en, y_pred_ts_en_tuned)
    ]
}).round(4).set_index("Dataset")

en_tuned_eval

##Random Forest

In [ ]:
# ============================
# 0) Split X/y (Random Forest)
# ============================
target = "actual_productivity_score"

X_tr_rf = df_tr.drop(columns=[target]).copy()
y_tr_rf = df_tr[target].copy()

X_ts_rf = df_ts.drop(columns=[target]).copy()
y_ts_rf = df_ts[target].copy()

# Ensure same columns/order (safe guard)
X_ts_rf = X_ts_rf.reindex(columns=X_tr_rf.columns)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ============================
# 1) Encoding (only for categoricals)
# Random Forest does NOT need scaling
# ============================
cat_cols = X_tr_rf.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X_tr_rf.columns if c not in cat_cols]  # includes bool/int/float

preprocess_rf = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

# ============================
# 2) Random Forest Model
# ============================
rf = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1,
    # Sensible defaults; you can tune later
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1
)

rf_model = Pipeline(steps=[
    ("prep", preprocess_rf),
    ("rf", rf)
])

# ============================
# 3) Fit + Predict
# ============================
rf_model.fit(X_tr_rf, y_tr_rf)

y_pred_tr_rf = rf_model.predict(X_tr_rf)
y_pred_ts_rf = rf_model.predict(X_ts_rf)

# ============================
# 4) Metrics (print)
# ============================
print("RANDOM FOREST — TRAIN")
print("R2:", r2_score(y_tr_rf, y_pred_tr_rf))
print("MAE:", mean_absolute_error(y_tr_rf, y_pred_tr_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_tr_rf, y_pred_tr_rf)))

print("\nRANDOM FOREST — TEST")
print("R2:", r2_score(y_ts_rf, y_pred_ts_rf))
print("MAE:", mean_absolute_error(y_ts_rf, y_pred_ts_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_ts_rf, y_pred_ts_rf)))

In [ ]:
# ============================
# 5) Clean evaluation table
# ============================
rf_eval = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [
        mean_absolute_error(y_tr_rf, y_pred_tr_rf),
        mean_absolute_error(y_ts_rf, y_pred_ts_rf)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_rf, y_pred_tr_rf)),
        np.sqrt(mean_squared_error(y_ts_rf, y_pred_ts_rf))
    ],
    "R²": [
        r2_score(y_tr_rf, y_pred_tr_rf),
        r2_score(y_ts_rf, y_pred_ts_rf)
    ]
}).round(4).set_index("Dataset")

rf_eval

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

param_distributions_fast = {
    "rf__n_estimators": [150, 250, 400],
    "rf__max_depth": [None, 10, 14, 18],
    "rf__min_samples_leaf": [1, 2, 4, 8],
    "rf__max_features": ["sqrt", 0.5, 0.8],
}

search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_distributions_fast,
    n_iter=12,            # ✅ much faster
    scoring="r2",
    cv=5,                 # ✅ still solid
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X_tr_rf, y_tr_rf)

best_rf_model = search.best_estimator_

print("Best CV R2:", search.best_score_)
print("Best Params:", search.best_params_)

y_pred_tr_rf_tuned = best_rf_model.predict(X_tr_rf)
y_pred_ts_rf_tuned = best_rf_model.predict(X_ts_rf)

rf_tuned_eval = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [
        mean_absolute_error(y_tr_rf, y_pred_tr_rf_tuned),
        mean_absolute_error(y_ts_rf, y_pred_ts_rf_tuned)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_rf, y_pred_tr_rf_tuned)),
        np.sqrt(mean_squared_error(y_ts_rf, y_pred_ts_rf_tuned))
    ],
    "R²": [
        r2_score(y_tr_rf, y_pred_tr_rf_tuned),
        r2_score(y_ts_rf, y_pred_ts_rf_tuned)
    ]
}).round(4).set_index("Dataset")

rf_tuned_eval

In [ ]:
from sklearn.ensemble import RandomForestRegressor

best_params = search.best_params_

final_rf = RandomForestRegressor(
    n_estimators=800,      # more trees for final stability
    random_state=42,
    n_jobs=-1,
    max_depth=best_params["rf__max_depth"],
    min_samples_leaf=best_params["rf__min_samples_leaf"],
    max_features=best_params["rf__max_features"],
)

final_rf_model = Pipeline(steps=[
    ("prep", preprocess_rf),
    ("rf", final_rf)
])

final_rf_model.fit(X_tr_rf, y_tr_rf)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# ---- Baseline RF eval table (you already have y_pred_tr_rf, y_pred_ts_rf) ----
rf_baseline_eval = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [
        mean_absolute_error(y_tr_rf, y_pred_tr_rf),
        mean_absolute_error(y_ts_rf, y_pred_ts_rf)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_rf, y_pred_tr_rf)),
        np.sqrt(mean_squared_error(y_ts_rf, y_pred_ts_rf))
    ],
    "R²": [
        r2_score(y_tr_rf, y_pred_tr_rf),
        r2_score(y_ts_rf, y_pred_ts_rf)
    ]
}).round(4).set_index("Dataset")

# ---- Final RF (800 trees) eval table (you fit final_rf_model but didn’t evaluate) ----
y_pred_tr_rf_final = final_rf_model.predict(X_tr_rf)
y_pred_ts_rf_final = final_rf_model.predict(X_ts_rf)

rf_final_eval = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [
        mean_absolute_error(y_tr_rf, y_pred_tr_rf_final),
        mean_absolute_error(y_ts_rf, y_pred_ts_rf_final)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_rf, y_pred_tr_rf_final)),
        np.sqrt(mean_squared_error(y_ts_rf, y_pred_ts_rf_final))
    ],
    "R²": [
        r2_score(y_tr_rf, y_pred_tr_rf_final),
        r2_score(y_ts_rf, y_pred_ts_rf_final)
    ]
}).round(4).set_index("Dataset")

In [ ]:
# ---------- helper: make any eval table -> indexed by Dataset ----------
def _ensure_dataset_index(df):
    df = df.copy()
    if "Dataset" in df.columns:
        df = df.set_index("Dataset")
    # if it's already indexed by Dataset, do nothing
    return df[["MAE", "RMSE", "R²"]]

# LR table (your evaluation_table)
lr_eval = _ensure_dataset_index(evaluation_table)
lr_eval["Model"] = "Linear Regression"

# RF baseline
rf_base = _ensure_dataset_index(rf_baseline_eval)
rf_base["Model"] = "Random Forest (Baseline)"

# RF tuned (CV)
rf_tuned = _ensure_dataset_index(rf_tuned_eval)
rf_tuned["Model"] = "Random Forest (Tuned CV)"

# RF final (800 trees)
rf_final = _ensure_dataset_index(rf_final_eval)
rf_final["Model"] = "Random Forest (Final 800 trees)"

# Combine
final_comparison = pd.concat([lr_eval, rf_base, rf_tuned, rf_final], axis=0)

# Multi-index: (Model, Dataset)
final_comparison = final_comparison.reset_index().set_index(["Model", "Dataset"])

# Order columns + round
final_comparison = final_comparison[["MAE", "RMSE", "R²"]].round(4)

final_comparison

In [ ]:
import pandas as pd
import numpy as np

# =============================
# 1. Get trained RF model
# =============================
rf_trained = best_rf_model.named_steps["rf"]

# =============================
# 2. Get preprocessor
# =============================
prep = best_rf_model.named_steps["prep"]

# =============================
# 3. Get feature names properly
# =============================

# categorical feature names after OneHotEncoding
cat_features = prep.named_transformers_["cat"].get_feature_names_out(cat_cols)

# numeric feature names (passthrough)
num_features = num_cols

# combine in correct order
feature_names = np.concatenate([cat_features, num_features])

# =============================
# 4. Get importances
# =============================
importances = rf_trained.feature_importances_

# sanity check (important)
assert len(feature_names) == len(importances)

# =============================
# 5. Create importance table
# =============================
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

# sort
feature_importance = (
    feature_importance
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

# =============================
# 6. Show top 20
# =============================
feature_importance.head(20)

In [ ]:
import matplotlib.pyplot as plt

# =============================
# Plot Top 20 Feature Importance (Horizontal)
# =============================

top_n = 20

# Select top features and reverse for proper horizontal display
top_features = feature_importance.head(top_n).iloc[::-1]

plt.figure(figsize=(10, 7))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")

plt.title("Top 20 Feature Importances — Random Forest")

plt.tight_layout()

plt.show()

##Gradient Boosting Regression

In [ ]:
# =========================================================
# 0) Split X/y (make copies like before)
# =========================================================
target = "actual_productivity_score"

X_tr_gb = df_tr.drop(columns=[target]).copy()
y_tr_gb = df_tr[target].copy()

X_ts_gb = df_ts.drop(columns=[target]).copy()
y_ts_gb = df_ts[target].copy()

# Ensure test has same columns/order (safe guard)
X_ts_gb = X_ts_gb.reindex(columns=X_tr_gb.columns)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# =========================================================
# 1) Preprocess: OneHot for categoricals, passthrough numeric
# (No scaling needed for tree boosting)
# =========================================================
cat_cols = X_tr_gb.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X_tr_gb.columns if c not in cat_cols]

preprocess_gb = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

# =========================================================
# 2A) HistGradientBoostingRegressor (recommended: faster)
# =========================================================
hgb = HistGradientBoostingRegressor(
    random_state=42,
    learning_rate=0.05,
    max_depth=None,        # trees grow with other constraints; you can tune
    max_iter=500,          # boosting rounds
    l2_regularization=0.0
)

hgb_model = Pipeline(steps=[
    ("prep", preprocess_gb),
    ("hgb", hgb)
])

# Fit + Predict
hgb_model.fit(X_tr_gb, y_tr_gb)
y_pred_tr_hgb = hgb_model.predict(X_tr_gb)
y_pred_ts_hgb = hgb_model.predict(X_ts_gb)

# Metrics table
hgb_eval = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [
        mean_absolute_error(y_tr_gb, y_pred_tr_hgb),
        mean_absolute_error(y_ts_gb, y_pred_ts_hgb)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_gb, y_pred_tr_hgb)),
        np.sqrt(mean_squared_error(y_ts_gb, y_pred_ts_hgb))
    ],
    "R²": [
        r2_score(y_tr_gb, y_pred_tr_hgb),
        r2_score(y_ts_gb, y_pred_ts_hgb)
    ]
}).round(4).set_index("Dataset")

print("HistGradientBoostingRegressor results:")
display(hgb_eval)

# =========================================================
# 2B) (Optional) Classic GradientBoostingRegressor (slower)
# =========================================================
gbr = GradientBoostingRegressor(
    random_state=42,
    learning_rate=0.05,
    n_estimators=500,
    max_depth=3
)

gbr_model = Pipeline(steps=[
    ("prep", preprocess_gb),
    ("gbr", gbr)
])

gbr_model.fit(X_tr_gb, y_tr_gb)
y_pred_tr_gbr = gbr_model.predict(X_tr_gb)
y_pred_ts_gbr = gbr_model.predict(X_ts_gb)

gbr_eval = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [
        mean_absolute_error(y_tr_gb, y_pred_tr_gbr),
        mean_absolute_error(y_ts_gb, y_pred_ts_gbr)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_gb, y_pred_tr_gbr)),
        np.sqrt(mean_squared_error(y_ts_gb, y_pred_ts_gbr))
    ],
    "R²": [
        r2_score(y_tr_gb, y_pred_tr_gbr),
        r2_score(y_ts_gb, y_pred_ts_gbr)
    ]
}).round(4).set_index("Dataset")

print("GradientBoostingRegressor results:")
display(gbr_eval)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# -----------------------------
# Parameter space (FAST + useful)
# -----------------------------
param_dist_hgb = {
    "hgb__learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
    "hgb__max_iter": [200, 400, 700],
    "hgb__max_depth": [None, 3, 5, 7],
    "hgb__min_samples_leaf": [20, 50, 100],
    "hgb__l2_regularization": [0.0, 0.1, 1.0, 5.0],
    "hgb__max_bins": [128, 255],
}

search_hgb = RandomizedSearchCV(
    estimator=hgb_model,                 # your pipeline: ("prep", preprocess_gb) + ("hgb", ...)
    param_distributions=param_dist_hgb,
    n_iter=15,                           # small but effective
    scoring="r2",
    cv=3,                                # much faster than 5
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search_hgb.fit(X_tr_gb, y_tr_gb)

print("\nBest CV R2:", search_hgb.best_score_)
print("Best Params:\n", search_hgb.best_params_)

best_hgb_model = search_hgb.best_estimator_

# -----------------------------
# Predict
# -----------------------------
y_pred_tr_hgb_tuned = best_hgb_model.predict(X_tr_gb)
y_pred_ts_hgb_tuned = best_hgb_model.predict(X_ts_gb)

# -----------------------------
# Evaluation table
# -----------------------------
hgb_tuned_eval = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [
        mean_absolute_error(y_tr_gb, y_pred_tr_hgb_tuned),
        mean_absolute_error(y_ts_gb, y_pred_ts_hgb_tuned)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_gb, y_pred_tr_hgb_tuned)),
        np.sqrt(mean_squared_error(y_ts_gb, y_pred_ts_hgb_tuned))
    ],
    "R²": [
        r2_score(y_tr_gb, y_pred_tr_hgb_tuned),
        r2_score(y_ts_gb, y_pred_ts_hgb_tuned)
    ]
}).round(4).set_index("Dataset")

hgb_tuned_eval

##XGBoost

In [ ]:
!pip install xgboost

In [ ]:
# =========================================================
# 0) Create copies (same pattern as before)
# =========================================================

target = "actual_productivity_score"

X_tr_xgb = df_tr.drop(columns=[target]).copy()
y_tr_xgb = df_tr[target].copy()

X_ts_xgb = df_ts.drop(columns=[target]).copy()
y_ts_xgb = df_ts[target].copy()

# Ensure same column order
X_ts_xgb = X_ts_xgb.reindex(columns=X_tr_xgb.columns)

In [ ]:
import numpy as np
import pandas as pd

from xgboost import XGBRegressor

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# =========================================================
# 1) Encoding
# =========================================================

cat_cols = X_tr_xgb.select_dtypes(include=["object", "category"]).columns.tolist()

num_cols = [c for c in X_tr_xgb.columns if c not in cat_cols]

preprocess_xgb = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop"
)

# =========================================================
# 2) XGBoost Model
# =========================================================

xgb = XGBRegressor(

    n_estimators=600,
    learning_rate=0.05,

    max_depth=6,

    subsample=0.8,
    colsample_bytree=0.8,

    random_state=42,
    n_jobs=-1,

    tree_method="hist"   # FAST
)

xgb_model = Pipeline(steps=[

    ("prep", preprocess_xgb),

    ("xgb", xgb)

])

# =========================================================
# 3) Fit
# =========================================================

xgb_model.fit(X_tr_xgb, y_tr_xgb)

# =========================================================
# 4) Predict
# =========================================================

y_pred_tr_xgb = xgb_model.predict(X_tr_xgb)

y_pred_ts_xgb = xgb_model.predict(X_ts_xgb)

# =========================================================
# 5) Evaluation table
# =========================================================

xgb_eval = pd.DataFrame({

    "Dataset": ["Train", "Test"],

    "MAE": [

        mean_absolute_error(y_tr_xgb, y_pred_tr_xgb),

        mean_absolute_error(y_ts_xgb, y_pred_ts_xgb)

    ],

    "RMSE": [

        np.sqrt(mean_squared_error(y_tr_xgb, y_pred_tr_xgb)),

        np.sqrt(mean_squared_error(y_ts_xgb, y_pred_ts_xgb))

    ],

    "R²": [

        r2_score(y_tr_xgb, y_pred_tr_xgb),

        r2_score(y_ts_xgb, y_pred_ts_xgb)

    ]

}).round(4).set_index("Dataset")

xgb_eval

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist_xgb = {

    "xgb__n_estimators": [300, 500, 700],

    "xgb__learning_rate": [0.01, 0.03, 0.05, 0.1],

    "xgb__max_depth": [3, 5, 6, 8],

    "xgb__subsample": [0.7, 0.8, 0.9],

    "xgb__colsample_bytree": [0.7, 0.8, 0.9],

    "xgb__min_child_weight": [1, 3, 5]

}

search_xgb = RandomizedSearchCV(

    xgb_model,

    param_distributions=param_dist_xgb,

    n_iter=15,

    scoring="r2",

    cv=3,

    random_state=42,

    n_jobs=-1,

    verbose=1

)

search_xgb.fit(X_tr_xgb, y_tr_xgb)

print("Best CV score:", search_xgb.best_score_)

print("Best params:", search_xgb.best_params_)

best_xgb_model = search_xgb.best_estimator_

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================
# Predictions (Tuned XGB)
# =========================
y_pred_tr_xgb_tuned = best_xgb_model.predict(X_tr_xgb)
y_pred_ts_xgb_tuned = best_xgb_model.predict(X_ts_xgb)

# =========================
# Proper Evaluation Table
# =========================
xgb_tuned_eval = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [
        mean_absolute_error(y_tr_xgb, y_pred_tr_xgb_tuned),
        mean_absolute_error(y_ts_xgb, y_pred_ts_xgb_tuned)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_xgb, y_pred_tr_xgb_tuned)),
        np.sqrt(mean_squared_error(y_ts_xgb, y_pred_ts_xgb_tuned))
    ],
    "R²": [
        r2_score(y_tr_xgb, y_pred_tr_xgb_tuned),
        r2_score(y_ts_xgb, y_pred_ts_xgb_tuned)
    ],
    # Optional but very useful for reporting
    "CV R² (RandomizedSearch)": [search_xgb.best_score_, search_xgb.best_score_]
}).round(4).set_index("Dataset")

xgb_tuned_eval

##LightGBM

In [ ]:
!pip install lightgbm

In [ ]:
# =========================================================
# 0) Copies like before
# =========================================================
target = "actual_productivity_score"

X_tr_lgb = df_tr.drop(columns=[target]).copy()
y_tr_lgb = df_tr[target].copy()

X_ts_lgb = df_ts.drop(columns=[target]).copy()
y_ts_lgb = df_ts[target].copy()

# Safety: same columns/order
X_ts_lgb = X_ts_lgb.reindex(columns=X_tr_lgb.columns)

In [ ]:
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# =========================================================
# 1) Encoding (OHE for category/object, passthrough numeric/bool)
# =========================================================
cat_cols = X_tr_lgb.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X_tr_lgb.columns if c not in cat_cols]

preprocess_lgb = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

# =========================================================
# 2) LightGBM model (strong default-ish settings)
# =========================================================
lgb = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lgb_model = Pipeline(steps=[
    ("prep", preprocess_lgb),
    ("lgb", lgb)
])

# =========================================================
# 3) Fit + Predict
# =========================================================
lgb_model.fit(X_tr_lgb, y_tr_lgb)

y_pred_tr_lgb = lgb_model.predict(X_tr_lgb)
y_pred_ts_lgb = lgb_model.predict(X_ts_lgb)

# =========================================================
# 4) Evaluation table
# =========================================================
lgb_eval = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [
        mean_absolute_error(y_tr_lgb, y_pred_tr_lgb),
        mean_absolute_error(y_ts_lgb, y_pred_ts_lgb)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_lgb, y_pred_tr_lgb)),
        np.sqrt(mean_squared_error(y_ts_lgb, y_pred_ts_lgb))
    ],
    "R²": [
        r2_score(y_tr_lgb, y_pred_tr_lgb),
        r2_score(y_ts_lgb, y_pred_ts_lgb)
    ]
}).round(4).set_index("Dataset")

lgb_eval

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist_lgb = {
    "lgb__n_estimators": [800, 1200, 2000],
    "lgb__learning_rate": [0.01, 0.03, 0.05, 0.08],
    "lgb__num_leaves": [15, 31, 63, 127],
    "lgb__max_depth": [-1, 4, 6, 8],
    "lgb__min_child_samples": [10, 20, 50, 100],
    "lgb__subsample": [0.7, 0.8, 0.9],
    "lgb__colsample_bytree": [0.7, 0.8, 0.9],
    "lgb__reg_alpha": [0.0, 0.1, 1.0],
    "lgb__reg_lambda": [0.0, 0.1, 1.0, 5.0],
}

search_lgb = RandomizedSearchCV(
    estimator=lgb_model,
    param_distributions=param_dist_lgb,
    n_iter=15,
    scoring="r2",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search_lgb.fit(X_tr_lgb, y_tr_lgb)

best_lgb_model = search_lgb.best_estimator_
print("Best CV R2:", search_lgb.best_score_)
print("Best params:", search_lgb.best_params_)

y_pred_tr_lgb_tuned = best_lgb_model.predict(X_tr_lgb)
y_pred_ts_lgb_tuned = best_lgb_model.predict(X_ts_lgb)

lgb_tuned_eval = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [
        mean_absolute_error(y_tr_lgb, y_pred_tr_lgb_tuned),
        mean_absolute_error(y_ts_lgb, y_pred_ts_lgb_tuned)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_lgb, y_pred_tr_lgb_tuned)),
        np.sqrt(mean_squared_error(y_ts_lgb, y_pred_ts_lgb_tuned))
    ],
    "R²": [
        r2_score(y_tr_lgb, y_pred_tr_lgb_tuned),
        r2_score(y_ts_lgb, y_pred_ts_lgb_tuned)
    ]
}).round(4).set_index("Dataset")

lgb_tuned_eval

##CatBoost

In [ ]:
!pip install catboost

In [ ]:
import numpy as np
import pandas as pd

target = "actual_productivity_score"

X_tr_cat = df_tr.drop(columns=[target]).copy()
y_tr_cat = df_tr[target].copy()

X_ts_cat = df_ts.drop(columns=[target]).copy()
y_ts_cat = df_ts[target].copy()

# identify categorical columns
cat_cols = X_tr_cat.select_dtypes(include=["category", "object"]).columns.tolist()

cat_cols

In [ ]:
from catboost import CatBoostRegressor
import pandas as pd

# -----------------------------
# 1) Make stress_level ORDINAL
# -----------------------------
# If stress_level values are like 1,2,...,10 but stored as category/strings/floats
# convert safely to numeric.
"""for df_ in [X_tr_cat, X_ts_cat]:
    df_["stress_level"] = pd.to_numeric(df_["stress_level"], errors="coerce")"""

# (Optional but recommended) If it's truly 1-10 integers:
for df_ in [X_tr_cat, X_ts_cat]:
    df_["stress_level"] = df_["stress_level"].round().astype("Int64")  # keeps NaN if any
    # If you are 100% sure there are no NaNs:
    # df_["stress_level"] = df_["stress_level"].astype(int)

# -----------------------------
# 2) CatBoost cat_features:
#    REMOVE stress_level
# -----------------------------
# Keep only nominal categorical columns here
cat_cols_nominal = [c for c in cat_cols if c != "stress_level"]

# Make sure nominal cat columns are string (CatBoost accepts int too, but string is safest)
for c in cat_cols_nominal:
    X_tr_cat[c] = X_tr_cat[c].astype(str)
    X_ts_cat[c] = X_ts_cat[c].astype(str)

# -----------------------------
# 3) Fit model
# -----------------------------
cat_model = CatBoostRegressor(
    iterations=800,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    random_seed=42,
    verbose=0
)

cat_model.fit(
    X_tr_cat,
    y_tr_cat,
    cat_features=cat_cols_nominal,   # ✅ stress_level removed
    eval_set=(X_ts_cat, y_ts_cat),
    verbose=False
)

In [ ]:
y_pred_tr_cat = cat_model.predict(X_tr_cat)
y_pred_ts_cat = cat_model.predict(X_ts_cat)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist_cat = {

    "depth": [4, 6, 8],

    "learning_rate": [0.01, 0.03, 0.05, 0.1],

    "iterations": [400, 600, 800],

    "l2_leaf_reg": [1, 3, 5, 7]
}

search_cat = RandomizedSearchCV(

    estimator=CatBoostRegressor(

        loss_function="RMSE",

        random_seed=42,

        verbose=0

    ),

    param_distributions=param_dist_cat,

    n_iter=15,

    scoring="r2",

    cv=3,

    n_jobs=-1,

    random_state=42
)

search_cat.fit(

    X_tr_cat,
    y_tr_cat,

    cat_features=cat_cols
)

best_cat_model = search_cat.best_estimator_

print("Best CV score:", search_cat.best_score_)
print("Best params:", search_cat.best_params_)

In [ ]:
y_pred_tr_cat_tuned = best_cat_model.predict(X_tr_cat)
y_pred_ts_cat_tuned = best_cat_model.predict(X_ts_cat)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
cat_tuned_eval = pd.DataFrame({

    "Dataset": ["Train", "Test"],

    "MAE": [
        mean_absolute_error(y_tr_cat, y_pred_tr_cat_tuned),
        mean_absolute_error(y_ts_cat, y_pred_ts_cat_tuned)
    ],

    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_cat, y_pred_tr_cat_tuned)),
        np.sqrt(mean_squared_error(y_ts_cat, y_pred_ts_cat_tuned))
    ],

    "R²": [
        r2_score(y_tr_cat, y_pred_tr_cat_tuned),
        r2_score(y_ts_cat, y_pred_ts_cat_tuned)
    ]

}).round(4).set_index("Dataset")

cat_tuned_eval

In [ ]:
cat_eval_final = cat_tuned_eval.copy()

cat_eval_final["Model"] = "CatBoost (Tuned)"

cat_eval_final = cat_eval_final.reset_index().set_index(["Model", "Dataset"])

##KNN

In [ ]:
import numpy as np
import pandas as pd

target = "actual_productivity_score"

X_tr_knn = df_tr.drop(columns=[target]).copy()
y_tr_knn = df_tr[target].copy()

X_ts_knn = df_ts.drop(columns=[target]).copy()
y_ts_knn = df_ts[target].copy()

# keep same columns/order
X_ts_knn = X_ts_knn.reindex(columns=X_tr_knn.columns)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

cat_cols = X_tr_knn.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X_tr_knn.columns if c not in cat_cols]

preprocess_knn = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop"
)

knn = KNeighborsRegressor(
    n_neighbors=15,
    weights="distance",
    metric="minkowski",   # Euclidean
    p=2
)

knn_model = Pipeline(steps=[
    ("prep", preprocess_knn),
    ("scale", StandardScaler(with_mean=False)),  # ✅ works with sparse after OHE
    ("knn", knn)
])

knn_model.fit(X_tr_knn, y_tr_knn)

y_pred_tr_knn = knn_model.predict(X_tr_knn)
y_pred_ts_knn = knn_model.predict(X_ts_knn)

knn_eval = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [
        mean_absolute_error(y_tr_knn, y_pred_tr_knn),
        mean_absolute_error(y_ts_knn, y_pred_ts_knn)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_knn, y_pred_tr_knn)),
        np.sqrt(mean_squared_error(y_ts_knn, y_pred_ts_knn))
    ],
    "R²": [
        r2_score(y_tr_knn, y_pred_tr_knn),
        r2_score(y_ts_knn, y_pred_ts_knn)
    ]
}).round(4).set_index("Dataset")

knn_eval

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_knn = {
    "knn__n_neighbors": [3, 5, 7, 11, 15, 21, 31],
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2]   # 1 = Manhattan, 2 = Euclidean
}

grid_knn = GridSearchCV(
    estimator=knn_model,
    param_grid=param_grid_knn,
    scoring="r2",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_knn.fit(X_tr_knn, y_tr_knn)

best_knn_model = grid_knn.best_estimator_

print("Best CV R2:", grid_knn.best_score_)
print("Best params:", grid_knn.best_params_)

y_pred_tr_knn_tuned = best_knn_model.predict(X_tr_knn)
y_pred_ts_knn_tuned = best_knn_model.predict(X_ts_knn)

knn_tuned_eval = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [
        mean_absolute_error(y_tr_knn, y_pred_tr_knn_tuned),
        mean_absolute_error(y_ts_knn, y_pred_ts_knn_tuned)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_knn, y_pred_tr_knn_tuned)),
        np.sqrt(mean_squared_error(y_ts_knn, y_pred_ts_knn_tuned))
    ],
    "R²": [
        r2_score(y_tr_knn, y_pred_tr_knn_tuned),
        r2_score(y_ts_knn, y_pred_ts_knn_tuned)
    ],
    "CV R² (GridSearch)": [grid_knn.best_score_, grid_knn.best_score_]
}).round(4).set_index("Dataset")

knn_tuned_eval

#Final Evaluation

In [ ]:
import pandas as pd

# ===============================
# Helper function (standardize)
# ===============================
def prepare_eval(df, model_name):

    df2 = df.copy()

    # if Dataset is column, make it index
    if "Dataset" in df2.columns:
        df2 = df2.set_index("Dataset")

    # keep only main metrics
    df2 = df2[["MAE", "RMSE", "R²"]]

    # add model column
    df2["Model"] = model_name

    return df2.reset_index()


# ===============================
# Prepare each model
# ===============================
tables = [

    prepare_eval(evaluation_table, "Linear Regression"),
    prepare_eval(ridge_eval, "Ridge Regression"),
    prepare_eval(lasso_eval, "Lasso Regression"),
    prepare_eval(en_tuned_eval, "Elastic Net (Tuned)"),

    prepare_eval(knn_tuned_eval, "KNN (Tuned)"),

    prepare_eval(rf_baseline_eval, "Random Forest (Baseline)"),
    prepare_eval(rf_final_eval, "Random Forest (Final)"),

    prepare_eval(gbr_eval, "Gradient Boosting"),
    prepare_eval(hgb_tuned_eval, "HistGradientBoosting (Tuned)"),

    prepare_eval(xgb_eval, "XGBoost (Baseline)"),
    prepare_eval(xgb_tuned_eval, "XGBoost (Tuned)"),

    prepare_eval(lgb_eval, "LightGBM (Baseline)"),
    prepare_eval(lgb_tuned_eval, "LightGBM (Tuned)"),

    prepare_eval(cat_tuned_eval, "CatBoost (Tuned)")
]

# ===============================
# Combine everything
# ===============================
final_evaluation = pd.concat(tables, ignore_index=True)

# set proper index
final_evaluation = final_evaluation.set_index(["Model", "Dataset"])

# round nicely
final_evaluation = final_evaluation.round(4)

final_evaluation

In [ ]:
final_evaluation_sorted = (
    final_evaluation
    .reset_index()
    .sort_values(by=["Dataset", "R²"], ascending=[True, False])
    .set_index(["Model", "Dataset"])
)

final_evaluation_sorted

#Evaluating the best model (CatBoost)

In [ ]:
import pandas as pd

feat_imp = pd.DataFrame({
    "feature": X_tr_cat.columns,
    "importance": cat_model.get_feature_importance(type="PredictionValuesChange")
}).sort_values("importance", ascending=False)

feat_imp.head(20)

In [ ]:
import matplotlib.pyplot as plt

top = feat_imp.head(20).iloc[::-1]  # reverse for horizontal plot
plt.figure(figsize=(8, 6))
plt.barh(top["feature"], top["importance"])
plt.title("CatBoost Feature Importance (PredictionValuesChange) - Top 20")
plt.xlabel("Importance")
plt.show()

##Permutation Importance

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

baseline_pred = cat_model.predict(X_ts_cat)
baseline_rmse = rmse(y_ts_cat, baseline_pred)

def permutation_importance(model, X, y, n_repeats=5, random_state=42):
    rng = np.random.RandomState(random_state)
    importances = []

    for col in X.columns:
        scores = []
        for _ in range(n_repeats):
            X_perm = X.copy()
            shuffled = X_perm[col].to_numpy(copy=True)
            rng.shuffle(shuffled)
            X_perm[col] = shuffled

            pred = model.predict(X_perm)
            scores.append(rmse(y, pred))

        importances.append(np.mean(scores) - baseline_rmse)

    return pd.DataFrame({"feature": X.columns, "perm_importance_rmse_increase": importances}) \
             .sort_values("perm_importance_rmse_increase", ascending=False)

perm_imp = permutation_importance(cat_model, X_ts_cat, y_ts_cat, n_repeats=5)
perm_imp.head(20)

##Shap

In [ ]:
import numpy as np
import pandas as pd
from catboost import Pool

# If you used this earlier:
# cat_cols_nominal = [c for c in cat_cols if c != "stress_level"]

# 1) Build Pool for test data
test_pool = Pool(
    data=X_ts_cat,
    label=y_ts_cat,
    cat_features=cat_cols_nominal   # or cat_cols if stress_level is categorical
)

# 2) Get SHAP values
shap_vals = cat_model.get_feature_importance(
    data=test_pool,
    type="ShapValues"
)

# shap_vals: (n_samples, n_features + 1), last col = expected value
shap_vals_features = shap_vals[:, :-1]

# 3) Global importance = mean absolute SHAP
shap_importance = np.mean(np.abs(shap_vals_features), axis=0)

shap_imp_df = pd.DataFrame({
    "feature": X_ts_cat.columns,
    "mean_abs_shap": shap_importance
}).sort_values("mean_abs_shap", ascending=False)

shap_imp_df.head(20)

In [ ]:
import matplotlib.pyplot as plt

top = shap_imp_df.head(20).iloc[::-1]
plt.figure(figsize=(8, 6))
plt.barh(top["feature"], top["mean_abs_shap"])
plt.title("CatBoost Mean |SHAP| Feature Importance (Top 20)")
plt.xlabel("Mean absolute SHAP value")
plt.show()

##Without job satisfaction score

In [ ]:
drop_cols = ["job_satisfaction_score"]

X_tr_no_js = X_tr_cat.drop(columns=drop_cols)
X_ts_no_js = X_ts_cat.drop(columns=drop_cols)

# update cat features list if needed (only if job_satisfaction_score is in it, usually not)
cat_feats_no_js = [c for c in cat_cols_nominal if c in X_tr_no_js.columns]

cat_model_no_js = CatBoostRegressor(
    iterations=800,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    random_seed=42,
    verbose=0
)

cat_model_no_js.fit(
    X_tr_no_js, y_tr_cat,
    cat_features=cat_feats_no_js,
    eval_set=(X_ts_no_js, y_ts_cat),
    verbose=False
)

from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

pred = cat_model_no_js.predict(X_ts_no_js)
print("R²:", r2_score(y_ts_cat, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_ts_cat, pred)))

##Drop interactions and dummy ecodings and test the model

In [ ]:
df_tr["stress_level"] = df_tr["stress_level"].astype("int64")
df_ts["stress_level"] = df_ts["stress_level"].astype("int64")

In [ ]:
df_tr.info()

In [ ]:
df_ts.info()

In [ ]:
df_tr_no_int = df_tr.copy()
df_ts_no_int = df_ts.copy()

In [ ]:
# interaction columns
interaction_cols = [c for c in df_tr_no_int.columns if "_x_work" in c]

# one-hot encoded job columns
job_dummy_cols = [
    "job_Finance",
    "job_Health",
    "job_IT",
    "job_Student",
    "job_Unemployed"
]

drop_cols = interaction_cols + job_dummy_cols

print("Dropping:", drop_cols)

In [ ]:
df_tr_no_int = df_tr_no_int.drop(columns=drop_cols)
df_ts_no_int = df_ts_no_int.drop(columns=drop_cols)

In [ ]:
print("Train shape:", df_tr_no_int.shape)
print("Test shape:", df_ts_no_int.shape)

In [ ]:
X_tr_cat_no_int = df_tr_no_int.drop(columns="actual_productivity_score")
y_tr_cat_no_int = df_tr_no_int["actual_productivity_score"]

X_ts_cat_no_int = df_ts_no_int.drop(columns="actual_productivity_score")
y_ts_cat_no_int = df_ts_no_int["actual_productivity_score"]

In [ ]:
cat_cols_nominal = [
    "gender",
    "job_type",
    "social_platform_preference"
]

# ensure string type
for c in cat_cols_nominal:
    X_tr_cat_no_int[c] = X_tr_cat_no_int[c].astype(str)
    X_ts_cat_no_int[c] = X_ts_cat_no_int[c].astype(str)

In [ ]:
from catboost import CatBoostRegressor

cat_model_no_int = CatBoostRegressor(
    iterations=800,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    random_seed=42,
    verbose=0
)

cat_model_no_int.fit(
    X_tr_cat_no_int,
    y_tr_cat_no_int,
    cat_features=cat_cols_nominal,
    eval_set=(X_ts_cat_no_int, y_ts_cat_no_int),
    verbose=False
)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

pred = cat_model_no_int.predict(X_ts_cat_no_int)

print("R²:", r2_score(y_ts_cat_no_int, pred))
print("MAE:", mean_absolute_error(y_ts_cat_no_int, pred))
print("RMSE:", np.sqrt(mean_absolute_error(y_ts_cat_no_int, pred)))

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Predictions
y_pred_tr_cat_no_int = cat_model_no_int.predict(X_tr_cat_no_int)
y_pred_ts_cat_no_int = cat_model_no_int.predict(X_ts_cat_no_int)

# Create evaluation table
cat_no_int_eval = pd.DataFrame({

    "Dataset": ["Train", "Test"],

    "MAE": [
        mean_absolute_error(y_tr_cat_no_int, y_pred_tr_cat_no_int),
        mean_absolute_error(y_ts_cat_no_int, y_pred_ts_cat_no_int)
    ],

    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_cat_no_int, y_pred_tr_cat_no_int)),
        np.sqrt(mean_squared_error(y_ts_cat_no_int, y_pred_ts_cat_no_int))
    ],

    "R²": [
        r2_score(y_tr_cat_no_int, y_pred_tr_cat_no_int),
        r2_score(y_ts_cat_no_int, y_pred_ts_cat_no_int)
    ]

}).round(4).set_index("Dataset")

cat_no_int_eval

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ================================
# Get feature importance
# ================================

# Importance values
importance = cat_model_no_int.get_feature_importance()

# Feature names
feature_names = X_tr_cat_no_int.columns

# Create dataframe
cat_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
})

# Sort
cat_importance = cat_importance.sort_values("Importance", ascending=False)

# Display
cat_importance

In [ ]:
plt.figure(figsize=(10,6))

plt.barh(
    cat_importance["Feature"],
    cat_importance["Importance"]
)

plt.gca().invert_yaxis()

plt.title("CatBoost Feature Importance (No Interaction Model)")
plt.xlabel("Importance")
plt.ylabel("Feature")

plt.show()

In [ ]:
import numpy as np
import pandas as pd
from catboost import Pool

# 1) Build Pool for NO-INTERACTION test data
test_pool_no_int = Pool(
    data=X_ts_cat_no_int,
    label=y_ts_cat_no_int,
    cat_features=cat_cols_nominal
)

# 2) Get SHAP values (n_samples, n_features + 1)
shap_vals_no_int = cat_model_no_int.get_feature_importance(
    data=test_pool_no_int,
    type="ShapValues"
)

# Last column = expected value, remove it
shap_vals_no_int_features = shap_vals_no_int[:, :-1]

# 3) Global importance = mean absolute SHAP
shap_importance_no_int = np.mean(np.abs(shap_vals_no_int_features), axis=0)

shap_imp_df_no_int = pd.DataFrame({
    "feature": X_ts_cat_no_int.columns,
    "mean_abs_shap": shap_importance_no_int
}).sort_values("mean_abs_shap", ascending=False)

shap_imp_df_no_int.head(20)

In [ ]:
import shap
import matplotlib.pyplot as plt
from catboost import Pool

# Build Pool
test_pool_no_int = Pool(
    data=X_ts_cat_no_int,
    label=y_ts_cat_no_int,
    cat_features=cat_cols_nominal
)

# Get SHAP values
shap_vals_no_int = cat_model_no_int.get_feature_importance(
    data=test_pool_no_int,
    type="ShapValues"
)

# Remove expected value column
shap_vals_no_int_features = shap_vals_no_int[:, :-1]


# SUMMARY PLOT
plt.figure()

shap.summary_plot(
    shap_vals_no_int_features,
    X_ts_cat_no_int,
    feature_names=X_ts_cat_no_int.columns
)

In [ ]:
comparison = pd.concat({

    "CatBoost (With Interaction)": cat_tuned_eval,

    "CatBoost (No Interaction)": cat_no_int_eval

})

comparison

#CatBoost with Job satisfaction score only

In [ ]:
df_tr.info()

In [ ]:
for col_df in [df_tr, df_ts]:
    # Convert 'stress_level' to numeric, then round and convert to nullable integer type
    # before converting to non-nullable int if no NaNs are present.
    col_df['stress_level'] = pd.to_numeric(col_df['stress_level'], errors='coerce').round().astype('Int64')
    col_df['stress_level'] = col_df['stress_level'].astype(int)

print("df_tr info after stress_level conversion:")
df_tr.info()
print("\ndf_ts info after stress_level conversion:")
df_ts.info()

In [ ]:
X_tr_cat_job = df_tr[["job_satisfaction_score"]]
y_tr_cat_job = df_tr["actual_productivity_score"]

X_ts_cat_job = df_ts[["job_satisfaction_score"]]
y_ts_cat_job = df_ts["actual_productivity_score"]

In [ ]:
X_tr_cat_job.info()

In [ ]:
from catboost import CatBoostRegressor

cat_model_job = CatBoostRegressor(
    iterations=800,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    random_seed=42,
    verbose=0
)

cat_model_job.fit(
    X_tr_cat_job,
    y_tr_cat_job,
    cat_features=[], # Empty list, as 'job_satisfaction_score' is a numeric feature
    eval_set=(X_ts_cat_job, y_ts_cat_job),
    verbose=False
)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Predictions
y_pred_tr_cat_job = cat_model_job.predict(X_tr_cat_job)
y_pred_ts_cat_job = cat_model_job.predict(X_ts_cat_job)

# Create evaluation table
cat_job_eval = pd.DataFrame({

    "Dataset": ["Train", "Test"],

    "MAE": [
        mean_absolute_error(y_tr_cat_job, y_pred_tr_cat_job),
        mean_absolute_error(y_ts_cat_job, y_pred_ts_cat_job)
    ],

    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_cat_job, y_pred_tr_cat_job)),
        np.sqrt(mean_squared_error(y_ts_cat_job, y_pred_ts_cat_job))
    ],

    "R²": [
        r2_score(y_tr_cat_job, y_pred_tr_cat_job),
        r2_score(y_ts_cat_job, y_pred_ts_cat_job)
    ]

}).round(4).set_index("Dataset")

cat_job_eval

In [ ]:
cat_job_eval_final = cat_job_eval.copy()
cat_job_eval_final["Model"] = "CatBoost (Job Satisfaction Only)"
cat_job_eval_final = cat_job_eval_final.reset_index().set_index(["Model", "Dataset"])
cat_job_eval_final

##Simple Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

lr_job = LinearRegression()

lr_job.fit(X_tr_cat_job, y_tr_cat_job)

y_pred_tr_lr_job = lr_job.predict(X_tr_cat_job)
y_pred_ts_lr_job = lr_job.predict(X_ts_cat_job)

lr_job_eval = pd.DataFrame({

    "Dataset": ["Train", "Test"],

    "MAE": [
        mean_absolute_error(y_tr_cat_job, y_pred_tr_lr_job),
        mean_absolute_error(y_ts_cat_job, y_pred_ts_lr_job)
    ],

    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_cat_job, y_pred_tr_lr_job)),
        np.sqrt(mean_squared_error(y_ts_cat_job, y_pred_ts_lr_job))
    ],

    "R²": [
        r2_score(y_tr_cat_job, y_pred_tr_lr_job),
        r2_score(y_ts_cat_job, y_pred_ts_lr_job)
    ]

}).round(4).set_index("Dataset")

lr_job_eval

##Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge

ridge_job = Ridge(alpha=1.0)

ridge_job.fit(X_tr_cat_job, y_tr_cat_job)

y_pred_tr_ridge_job = ridge_job.predict(X_tr_cat_job)
y_pred_ts_ridge_job = ridge_job.predict(X_ts_cat_job)

ridge_job_eval = pd.DataFrame({

    "Dataset": ["Train", "Test"],

    "MAE": [
        mean_absolute_error(y_tr_cat_job, y_pred_tr_ridge_job),
        mean_absolute_error(y_ts_cat_job, y_pred_ts_ridge_job)
    ],

    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_cat_job, y_pred_tr_ridge_job)),
        np.sqrt(mean_squared_error(y_ts_cat_job, y_pred_ts_ridge_job))
    ],

    "R²": [
        r2_score(y_tr_cat_job, y_pred_tr_ridge_job),
        r2_score(y_ts_cat_job, y_pred_ts_ridge_job)
    ]

}).round(4).set_index("Dataset")

ridge_job_eval

##Lasso Regression

In [ ]:
from sklearn.linear_model import Lasso

lasso_job = Lasso(alpha=0.01)

lasso_job.fit(X_tr_cat_job, y_tr_cat_job)

y_pred_tr_lasso_job = lasso_job.predict(X_tr_cat_job)
y_pred_ts_lasso_job = lasso_job.predict(X_ts_cat_job)

lasso_job_eval = pd.DataFrame({

    "Dataset": ["Train", "Test"],

    "MAE": [
        mean_absolute_error(y_tr_cat_job, y_pred_tr_lasso_job),
        mean_absolute_error(y_ts_cat_job, y_pred_ts_lasso_job)
    ],

    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_cat_job, y_pred_tr_lasso_job)),
        np.sqrt(mean_squared_error(y_ts_cat_job, y_pred_ts_lasso_job))
    ],

    "R²": [
        r2_score(y_tr_cat_job, y_pred_tr_lasso_job),
        r2_score(y_ts_cat_job, y_pred_ts_lasso_job)
    ]

}).round(4).set_index("Dataset")

lasso_job_eval

##ElasticNet Regression

In [ ]:
from sklearn.linear_model import ElasticNet

elastic_job = ElasticNet(alpha=0.01, l1_ratio=0.5)

elastic_job.fit(X_tr_cat_job, y_tr_cat_job)

y_pred_tr_elastic_job = elastic_job.predict(X_tr_cat_job)
y_pred_ts_elastic_job = elastic_job.predict(X_ts_cat_job)

elastic_job_eval = pd.DataFrame({

    "Dataset": ["Train", "Test"],

    "MAE": [
        mean_absolute_error(y_tr_cat_job, y_pred_tr_elastic_job),
        mean_absolute_error(y_ts_cat_job, y_pred_ts_elastic_job)
    ],

    "RMSE": [
        np.sqrt(mean_squared_error(y_tr_cat_job, y_pred_tr_elastic_job)),
        np.sqrt(mean_squared_error(y_ts_cat_job, y_pred_ts_elastic_job))
    ],

    "R²": [
        r2_score(y_tr_cat_job, y_pred_tr_elastic_job),
        r2_score(y_ts_cat_job, y_pred_ts_elastic_job)
    ]

}).round(4).set_index("Dataset")

elastic_job_eval

In [ ]:
def prepare(df, name):

    df2 = df.copy()

    df2["Model"] = name

    return df2.reset_index()

final_linear_models = pd.concat([

    prepare(lr_job_eval, "Linear Regression"),

    prepare(ridge_job_eval, "Ridge"),

    prepare(lasso_job_eval, "Lasso"),

    prepare(elastic_job_eval, "Elastic Net")

])

final_linear_models = (
    final_linear_models
    .set_index(["Model", "Dataset"])
    .round(4)
)

final_linear_models

In [ ]:
residuals = y_ts_cat_job - cat_model_job.predict(X_ts_cat_job)

plt.scatter(X_ts_cat_job, residuals)
plt.axhline(0)

#GAM

In [ ]:
# If pygam is not installed:
!pip install pygam

In [ ]:
import numpy as np
import pandas as pd

from pygam import LinearGAM, s
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# pygam needs numpy arrays
Xtr = X_tr_cat_job.values  # shape (n, 1)
ytr = y_tr_cat_job.values

Xts = X_ts_cat_job.values
yts = y_ts_cat_job.values

# Baseline GAM: smooth term on job_satisfaction_score
gam = LinearGAM(s(0, n_splines=10)).fit(Xtr, ytr)

# Predict
y_pred_tr_gam = gam.predict(Xtr)
y_pred_ts_gam = gam.predict(Xts)

# Eval table
gam_eval = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [
        mean_absolute_error(ytr, y_pred_tr_gam),
        mean_absolute_error(yts, y_pred_ts_gam)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(ytr, y_pred_tr_gam)),
        np.sqrt(mean_squared_error(yts, y_pred_ts_gam))
    ],
    "R²": [
        r2_score(ytr, y_pred_tr_gam),
        r2_score(yts, y_pred_ts_gam)
    ]
}).round(4).set_index("Dataset")

gam_eval

In [ ]:
gam.summary()

In [ ]:
from pygam import LinearGAM, s

# Build grid
splines_grid = [10, 12, 15, 20, 25]
lam_grid = np.logspace(-3, 3, 13)

best_gam = None
best_test_r2 = -np.inf
best_params = None

for n_splines in splines_grid:
    for lam in lam_grid:
        g = LinearGAM(s(0, n_splines=n_splines), lam=lam).fit(Xtr, ytr)
        pred = g.predict(Xts)
        r2 = r2_score(yts, pred)
        if r2 > best_test_r2:
            best_test_r2 = r2
            best_gam = g
            best_params = (n_splines, lam)

print("Best params (n_splines, lam):", best_params)
print("Best Test R²:", round(best_test_r2, 4))

# Evaluate best model
y_pred_tr_gam_tuned = best_gam.predict(Xtr)
y_pred_ts_gam_tuned = best_gam.predict(Xts)

gam_tuned_eval = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [
        mean_absolute_error(ytr, y_pred_tr_gam_tuned),
        mean_absolute_error(yts, y_pred_ts_gam_tuned)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(ytr, y_pred_tr_gam_tuned)),
        np.sqrt(mean_squared_error(yts, y_pred_ts_gam_tuned))
    ],
    "R²": [
        r2_score(ytr, y_pred_tr_gam_tuned),
        r2_score(yts, y_pred_ts_gam_tuned)
    ]
}).round(4).set_index("Dataset")

gam_tuned_eval

In [ ]:
import matplotlib.pyplot as plt

# grid of X values across observed range
x_grid = np.linspace(Xtr.min(), Xtr.max(), 200).reshape(-1, 1)
y_grid = best_gam.predict(x_grid)

plt.figure(figsize=(7,5))
plt.scatter(Xtr[:,0], ytr, alpha=0.2)
plt.plot(x_grid[:,0], y_grid)
plt.xlabel("Job Satisfaction Score")
plt.ylabel("Actual Productivity Score")
plt.title("GAM Fit: Productivity vs Job Satisfaction")
plt.tight_layout()
plt.show()

In [ ]:
gam_eval_final = gam_tuned_eval.copy()
gam_eval_final["Model"] = "GAM (Tuned)"
gam_eval_final = gam_eval_final.reset_index().set_index(["Model", "Dataset"])
gam_eval_final

In [ ]:
import pandas as pd

# helper function
def prepare_eval(df, model_name):

    df2 = df.copy()

    if "Dataset" in df2.columns:
        df2 = df2.set_index("Dataset")

    df2 = df2[["MAE", "RMSE", "R²"]]

    df2["Model"] = model_name

    return df2.reset_index()


# combine
tables = [

    prepare_eval(cat_job_eval, "CatBoost (Job Satisfaction Only)"),

    prepare_eval(lr_job_eval, "Linear Regression"),

    prepare_eval(ridge_job_eval, "Ridge"),

    prepare_eval(lasso_job_eval, "Lasso"),

    prepare_eval(elastic_job_eval, "Elastic Net"),
]

final_evaluation = pd.concat(tables, ignore_index=True)

final_evaluation = (
    final_evaluation
    .set_index(["Model", "Dataset"])
    .round(4)
)

final_evaluation

In [ ]:
import pandas as pd

# helper function
def prepare_eval(df, model_name):

    df2 = df.copy()

    if "Dataset" in df2.columns:
        df2 = df2.set_index("Dataset")

    df2 = df2[["MAE", "RMSE", "R²"]]

    df2["Model"] = model_name

    return df2.reset_index()


# combine
tables = [

    prepare_eval(lr_job_eval, "Linear Regression"),

    prepare_eval(ridge_job_eval, "Ridge"),

    prepare_eval(lasso_job_eval, "Lasso"),

    prepare_eval(elastic_job_eval, "Elastic Net"),

    prepare_eval(cat_job_eval, "CatBoost (Job Satisfaction Only)"),

    prepare_eval(gam_tuned_eval, "GAM (Tuned)")
]

final_evaluation = pd.concat(tables, ignore_index=True)

final_evaluation = (
    final_evaluation
    .set_index(["Model", "Dataset"])
    .round(4)
)

final_evaluation